# Introduction

## 1. What is Collaborative Filtering?
Collaborative Filtering (CF) is a fundamental technique in recommender systems that automates the process of predicting a user's preference or rating for a specific item based on the observed preferences of a community of other users. Unlike content-based filtering, which relies on the intrinsic properties of items (e.g., genre, color, keywords), collaborative filtering relies solely on past user-item interactions. The foundational assumption is that if User A and User B have historically agreed on the quality of several items, they are likely to agree on a new, unseen item in the future.

## 2. Is it about User-to-User, Item-to-Item, or User-to-Item Similarity?
Collaborative Filtering encompasses **all three perspectives**, depending on the specific algorithmic approach adopted:
- **User-to-User Similarity (Memory-Based CF):** This approach directly compares the rating vectors of users. If the system needs to recommend an item to User $u$, it finds $k$ other users whose rating history is most similar to $u$'s (neighbors).
- **Item-to-Item Similarity (Memory-Based CF):** This approach transposes the matrix and compares items based on how users have rated them. If a user liked Item $i$, the system recommends other items that have the most similar rating profile to $i$. This was famously popularized by Amazon's "Customers Who Bought This Item Also Bought" feature.
- **User-to-Item Relationship (Model-Based CF / Matrix Factorization):** This approach does not compute explicit similarity between raw rows or columns. Instead, it models the interaction as an inner product of latent factors—a **User-to-Item** affinity. It discovers abstract features that explain the observed ratings, such that the predicted rating $\hat{r}_{ui}$ is a function of the user's preference vector and the item's characteristic vector.

## 3. When Exactly is the Proper Case and Time to Use Collaborative Filtering?
Collaborative Filtering is the appropriate methodology under the following specific conditions:
- **Sparse User Feedback:** You have a large user base and a large item catalog, but each individual user has only interacted with a tiny fraction of the total items.
- **Subjective or Aesthetic Domains:** The items lack objective, machine-readable attributes (e.g., movies, music, jokes, handmade crafts). For example, you cannot analyze the "pixel data" of a movie to determine if it is "funny"; you must rely on what other people who laughed at the same things thought.
- **Cross-Category Discovery:** You want to surprise the user with recommendations from categories they have never visited (serendipity). For instance, a user who likes *Blade Runner* might like a specific Philip K. Dick novel, an association only visible through overlapping fanbases, not through text analysis.

## 4. Where Can We Implement Collaborative Filtering?
Collaborative Filtering is implemented across a wide spectrum of digital services where user decision fatigue is high:
- **E-commerce Platforms:** Amazon, eBay, and Etsy use it for product recommendations on detail pages and checkout cross-sells.
- **Streaming Media Services:** Netflix (movie suggestions), Spotify (Discover Weekly playlist), and YouTube (video recommendations) rely heavily on CF to keep users engaged by surfacing deep catalog content.
- **Social Networks:** LinkedIn ("People You May Know"), Facebook (Friend Suggestions), and Twitter (Who to Follow) treat "following a user" as the item interaction.
- **Digital Advertising:** Ad exchanges use CF to predict Click-Through Rate (CTR) for a specific user-ad pair based on similar user cohorts.

## 5. Why Should We Use Collaborative Filtering?
The primary justification for using Collaborative Filtering is **domain independence and discovery**. We should use it because:
1.  **No Feature Engineering Required:** It eliminates the need for subject matter experts to manually tag every song with "distorted guitar" or every book with "unreliable narrator." The system learns these latent descriptors from the data itself.
2.  **Quality of Insight:** It captures nuanced, hard-to-quantify preferences (e.g., "Movies with a bittersweet ending").
3.  **Scalability of Model Training:** Modern matrix factorization algorithms are highly parallelizable and can handle matrices with hundreds of millions of rows and columns efficiently using Stochastic Gradient Descent (SGD) or Alternating Least Squares (ALS).

## 6. Step-by-Step Mathematical Workflow with Formulation
When we have the data, specifically a set of triplets $(u, i, r_{ui})$ where $u$ is user ID, $i$ is item ID, and $r_{ui}$ is the interaction strength (explicit rating or implicit count). We work with **Model-Based Collaborative Filtering (Matrix Factorization)** as follows:

### **Step 1: Define the Interaction Matrix $R$**
We construct a sparse matrix $R \in \mathbb{R}^{m \times n}$, where $m$ is the number of users and $n$ is the number of items. Most entries $r_{ui}$ are missing (NaN).

### **Step 2: Mathematical Model Definition (Hypothesis)**
We assume each user $u$ can be represented by a latent vector $p_u \in \mathbb{R}^k$ and each item $i$ by a latent vector $q_i \in \mathbb{R}^k$. Here, $k$ is the number of latent factors (dimensionality of the abstract space, e.g., $k=50$ or $100$).
The predicted rating $\hat{r}_{ui}$ is modeled as the dot product (interaction) between the user and item vectors:
$$
\hat{r}_{ui} = q_i^T p_u = \sum_{f=1}^{k} q_{if} \cdot p_{uf}
$$

### **Step 3: Define the Loss Function (Objective)**
We need to minimize the difference between the observed ratings $r_{ui}$ and the predicted ratings $\hat{r}_{ui}$. To prevent overfitting to the sparse data, we introduce **L2 Regularization**. The objective function $\mathcal{L}$ to minimize is:
$$
\mathcal{L} = \min_{p^*, q^*} \sum_{(u,i) \in \mathcal{K}} \left( r_{ui} - q_i^T p_u \right)^2 + \lambda \left( \|q_i\|^2 + \|p_u\|^2 \right)
$$

Where:
- $\mathcal{K}$ is the set of known (user, item) pairs in the training data.
- $\lambda$ is the regularization hyperparameter controlling the penalty on vector magnitude.

### **Step 4: Optimization via Stochastic Gradient Descent (SGD)**
We iterate over each known rating $r_{ui}$ in the training set. We compute the prediction error:
$$
e_{ui} = r_{ui} - q_i^T p_u
$$

We then compute the gradients of the loss with respect to the parameters and update them in the opposite direction of the gradient. **Update for User Vector $p_u$:**
$$
p_u \leftarrow p_u + \eta \cdot (e_{ui} \cdot q_i - \lambda \cdot p_u)
$$

**Update for Item Vector $q_i$:**
$$
q_i \leftarrow q_i + \eta \cdot (e_{ui} \cdot p_u - \lambda \cdot q_i)
$$

Where $\eta$ is the learning rate.

### **Step 5: Prediction for Missing Values**
After convergence (or a fixed number of epochs), we reconstruct the full matrix $\hat{R}$. For a user $u$ and an unseen item $j$, the final predicted score is:
$$
\hat{r}_{uj} = q_j^T p_u
$$
We then recommend the top-$N$ items with the highest $\hat{r}_{uj}$ values that the user has not yet interacted with.

# Module Loading

In [1]:
import gdown
import os
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pylab import rcParams
import pandas as pd
import numpy as np
from google.colab import drive
from warnings import filterwarnings
import duckdb
import pandas as pd
from typing import Optional, Union, List, Dict, Any, ContextManager
from pathlib import Path
from contextlib import contextmanager
filterwarnings('ignore')

In [2]:
# Modern Professional Color Palette
modern_colors = [
    "#1f77b4",   # Vibrant Blue (Primary)
    "#ff7f0e",   # Bright Orange (Accent/Comparison)
    "#2ca02c",   # Fresh Green (Success/Positive)
    "#d62728",   # Soft Red (Alert/Warning)
    "#9467bd",   # Elegant Purple
    "#8c564b",   # Warm Brown
    "#e377c2",   # Pink
    "#7f7f7f",   # Neutral Gray
    "#bcbd22",   # Olive/Yellow-Green
    "#17becf"    # Cyan/Teal
]

# 2. Main Style Dictionary
modern_light_style = {
    # Background - Clean and bright
    "figure.facecolor": "#ffffff",
    "axes.facecolor": "#f8f9fa",
    "savefig.facecolor": "#ffffff",

    # Grid - Very subtle and non-distracting
    "axes.grid": True,
    "grid.color": "#e6e8eb",
    "grid.linestyle": "--",
    "grid.linewidth": 0.8,
    "axes.grid.which": "both",

    # Typography
    "text.color": "#1f2937",
    "axes.labelcolor": "#1f2937",
    "xtick.color": "#374151",
    "ytick.color": "#374151",
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.titlepad": 18,
    "font.size": 12,
    "font.family": "sans-serif", # You can change to 'Arial', 'Helvetica', etc.

    # Spines - Clean and minimal
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": True,
    "axes.spines.bottom": True,
    "axes.edgecolor": "#4b5563",
    "axes.linewidth": 1.2,

    # Lines and markers
    "axes.prop_cycle": plt.cycler(color=modern_colors),
    "lines.linewidth": 2.5,
    "lines.markersize": 7,
    "lines.markeredgewidth": 0.8,

    # Patches (bars, areas, etc.)
    "patch.edgecolor": "#ffffff",
    "patch.linewidth": 0.8,

    # Legend
    "legend.frameon": False,
    "legend.loc": "best",
    "legend.fontsize": 11,
}

plt.style.use('default')
sns.set_theme(style="whitegrid", rc=modern_light_style)
plt.rcParams.update(modern_light_style)
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=modern_colors)

In [3]:
class DuckDBManager:
    """
    Manager class for DuckDB connections and queries.

    Example:
        db = DuckDBManager('mydb.duckdb')
        df = db.query("SELECT * FROM users")
        db.close()

        # Or use context manager:
        with DuckDBManager('mydb.duckdb') as db:
            df = db.query("SELECT * FROM users")
    """

    def __init__(self,
                 db_path: Union[str, Path],
                 read_only: bool = True,
                 threads: Optional[int] = None,
                 memory_limit: Optional[str] = '2GB'):
        """
        Initialize DuckDB manager.

        Args:
            db_path: Path to database file (use ':memory:' for in-memory DB)
            read_only: Open in read-only mode
            threads: Number of CPU threads to use
            memory_limit: Memory limit (e.g., '4GB', '1TB')
        """
        self.db_path = str(db_path)
        self.read_only = read_only
        self.conn = None

        # Create connection
        self._create_connection(threads, memory_limit)

    def _create_connection(self, threads: Optional[int] = None,
                          memory_limit: Optional[str] = None):
        """Create DuckDB connection with optional settings."""
        self.conn = duckdb.connect(database=self.db_path, read_only=self.read_only)

        # Configure settings
        if threads:
            self.conn.execute(f"SET threads = {threads}")
        if memory_limit:
            self.conn.execute(f"SET memory_limit = '{memory_limit}'")

    def query(self,
              query: str,
              params: Optional[Union[List, Dict, tuple]] = None,
              fetch_size: Optional[int] = None) -> pd.DataFrame:
        """
        Execute query and return DataFrame.

        Args:
            query: SQL query string
            params: Query parameters
            fetch_size: Number of rows to fetch (None for all)

        Returns:
            pandas DataFrame
        """
        if self.conn is None:
            raise ValueError("Connection is closed. Please reconnect.")

        try:
            if params is not None:
                result = self.conn.execute(query, params)
            else:
                result = self.conn.execute(query)

            if fetch_size:
                return result.fetch_df_chunk(fetch_size)
            return result.fetchdf()

        except Exception as e:
            raise Exception(f"Query failed: {e}\nQuery: {query}")

    def query_arrow(self, query: str, params: Optional[Union[List, Dict]] = None):
        """Execute query and return Arrow table (faster for large datasets)."""
        if params is not None:
            return self.conn.execute(query, params).fetch_arrow_table()
        return self.conn.execute(query).fetch_arrow_table()

    def register_dataframe(self, name: str, df: pd.DataFrame):
        """Register pandas DataFrame as temporary table (table_view)."""
        self.conn.register(name, df)

    def ListedTable(self):
        tables = self.conn.execute("SELECT table_name FROM duckdb_tables()").df()
        data = list()
        for table in tables['table_name']:
            count = self.conn.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
            data.append({'table_name': table, 'row_count': count})
        info_df = pd.DataFrame(data)
        info_df = info_df.sort_values(by='row_count', ascending=False)
        info_df['row_count'] = info_df['row_count'].apply(lambda x: f"{x:,}")
        display(info_df)
        return info_df

    def execute(self, query: str, params: Optional[Union[List, Dict]] = None):
        """Execute query without returning results."""
        if params is not None:
            self.conn.execute(query, params)
        else:
            self.conn.execute(query)

    def table_exists(self, table_name: str) -> bool:
        """Check if table exists in database."""
        result = self.query(
            "SELECT COUNT(*) FROM information_schema.tables WHERE table_name = ?",
            [table_name]
        )
        return result.iloc[0, 0] > 0

    def get_tables(self) -> List[str]:
        """Get list of all tables in database."""
        df = self.query("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'")
        return df['table_name'].tolist()

    def get_schema(self, table_name: str) -> pd.DataFrame:
        """Get schema information for a table."""
        return self.query(f"DESCRIBE {table_name}")

    def close(self):
        """Close database connection."""
        if self.conn:
            self.conn.close()
            self.conn = None

    def __enter__(self):
        """Context manager entry."""
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Context manager exit."""
        self.close()


In [4]:
@contextmanager
def duckdb_connection(db_path: Union[str, Path] = './dbprocess.duckdb',
                      read_only: bool = False,
                      **kwargs) -> DuckDBManager:
    """
    Context manager for DuckDB connection.

    Example:
        with duckdb_connection('mydb.duckdb') as db:
            df = db.query("SELECT * FROM users")
    """
    db = DuckDBManager(db_path, read_only, **kwargs)
    try:
        yield db
    finally:
        db.close()

# Data Loading

## Download Data

In [5]:
def glink(uid : str, 
          thepath : str, 
          verbose : bool = False,
         ) -> str:
    coks = re.search(r'[-\w]{25,}', uid)
    file_id = coks.group(0) if coks else uid
    gdown.download(id=file_id, output=str(thepath), quiet = not verbose)
    thepath = os.path.realpath(thepath)
    
    # Verify the download isn't a tiny HTML
    if os.path.exists(thepath):
        file_size_kb = os.path.getsize(thepath) / 1024
        if file_size_kb < 100:
            print(f"Warning: {thepath} is very small ({file_size_kb:.2f} KB).")
            print("Check if the Google Drive link is set to 'Anyone with the link'.")
    return thepath

In [6]:
# Data GA4
url_ga4 = '1GRTBYqA0iKLLFflzcfNo1ctVWzis872Q'
dbfilename = './DB_ga4_ecommerce.duckdb'
ga4dbpath = glink(url_ga4, dbfilename)

In [7]:
#Data FPgrowth Prediction
url_fpg = '1YZN36UfzIWZ_Fy6ManAnQxeYjSBHA8Fn'
fpgname = './20260418_fpgrowth_prediction.zip'
fpgdbpath = glink(url_fpg, fpgname)

In [8]:
print(f'GApath : {ga4dbpath}.\n')
print(f'FP Growth Prediction : {fpgdbpath}.')

GApath : /kaggle/working/DB_ga4_ecommerce.duckdb.

FP Growth Prediction : /kaggle/working/20260418_fpgrowth_prediction.zip.


In [9]:
db = DuckDBManager(db_path = ga4dbpath)
_ = db.ListedTable()

mas_train = db.query('SELECT * FROM FinalMasterData')

,table_name,row_count
0,FinalMasterData,"1,014,426"
13,MasterTrainData,"1,014,426"
2,FinalMasterData_train,"811,540"
1,FinalMasterData_test,"202,886"
14,UserFeature_Analysis,"47,474"
3,fpgrowth_transactions,"46,375"
12,mapping_product_name,429
9,mapping_primary_country,109
7,mapping_department_id,81
5,mapping_brand,7


In [10]:
predpath = os.path.dirname(ga4dbpath)
if 'predpath' in locals() or 'predpath' in globals():
    print(f"Searching for ZIP files in: {predpath}")
    zip_files = [f for f in os.listdir(predpath) if f.endswith('.zip')]

    if zip_files:
        print("Found zip files:")
        for jf in zip_files:
            print(os.path.join(predpath, jf))
    else:
        print("No zip files found in this directory.")
else:
    print("Error: 'predpath' variable is not defined. Please ensure the previous cell setting predpath was executed.")

Searching for ZIP files in: /kaggle/working
Found zip files:
/kaggle/working/20260418_fpgrowth_prediction.zip


In [11]:
from shutil import copy
from zipfile import ZipFile

destination_dir = '/kaggle/working/fpgrowth_data'
os.makedirs(destination_dir, exist_ok = True)
all_extracted_json_data = list()

print(f"Processing ZIP files from {predpath}.")
for zip_file_name in zip_files:
    src_zip_path = os.path.join(predpath, zip_file_name)
    dest_zip_path = os.path.join(destination_dir, zip_file_name)

    if not os.path.exists(src_zip_path):
        print(f"Warning: Source ZIP file not found: {src_zip_path}. Skipping.")
        continue
    copy(src_zip_path, dest_zip_path)
    with ZipFile(dest_zip_path, 'r') as zip_ref:
        zip_ref.extractall(destination_dir)

Processing ZIP files from /kaggle/working.


In [12]:
import json
jsonfile = [os.path.join(destination_dir, f) for f in os.listdir(destination_dir) if f.endswith('.json')]
choosejson = jsonfile[-1]

with open(choosejson, 'r') as f:
    fpgrowthdata = json.load(f)

In [13]:
from itertools import islice

def dictslice(jsonfile:dict,
              count : int = 10,
              show : bool = True) -> dict:
    Data = dict(islice(jsonfile.items(), int(count)))
    if show:
        print(Data)
    return Data

a = dictslice(fpgrowthdata, 3)

{'1000684.125': [{'items': ['9196908'], 'support': 14}, {'items': ['9199084'], 'support': 44}], '1001326.125': [{'items': ['9197398'], 'support': 34}, {'items': ['9197946'], 'support': 10}], '1010983.25': [{'items': ['9196832'], 'support': 25}]}


In [14]:
def Json2Dataframe(jsondata : dict) -> pd.DataFrame:
    records = ({"group_id": group_id,
                "items"   : entry["items"],
                "support" : entry["support"]}
        for group_id, entries in jsondata.items()
        for entry in entries)
    data = pd.DataFrame.from_records(records)
    data["items"] = data["items"].apply(
        lambda x: ", ".join(x) if isinstance(x, (list, set)) else x)
    return data

DataFPGpredict = Json2Dataframe(fpgrowthdata)
display(DataFPGpredict.describe(include='object'))
print('\n\n')
display(DataFPGpredict.describe())

,group_id,items
count,27290,27290
unique,10631,677
top,59061040.0,9188192
freq,10,1958


,support
count,27290.000000
mean,48.772774
std,37.048558
min,8.000000
25%,19.000000
50%,38.000000
75%,72.000000
max,178.000000


In [15]:
print(mas_train.info())
#print(mas_train.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1014426 entries, 0 to 1014425
Data columns (total 42 columns):
 #   Column                             Non-Null Count    Dtype  
---  ------                             --------------    -----  
 0   user_id                            1014426 non-null  float32
 1   product_id                         1014426 non-null  object 
 2   department_id                      1014426 non-null  int32  
 3   aisle_id                           1014426 non-null  int32  
 4   brand                              1014426 non-null  int32  
 5   category3                          1014426 non-null  int32  
 6   avg_interaction_price              1014426 non-null  float32
 7   view_count                         1014426 non-null  int32  
 8   add_to_cart_count                  1014426 non-null  int32  
 9   begin_checkout_count               1014426 non-null  int32  
 10  purchase_count                     1014426 non-null  int32  
 11  total_quantity          

## Notes

For variable data, we can refer to `DataFPGpredict` and `mas_train`!!!

# **Collaborative Filtering**

## Cython Optimized

In [16]:
os.makedirs('./cy', exist_ok = True)

In [17]:
%%writefile ./cy/_cy_evaluate.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_evaluate.pyx
# ROC-AUC computation from pre-computed item ranks for arycolbring.
# Parallelised per-user with OpenMP prange.

from libc.stdio cimport fprintf, stderr
from cython.parallel cimport prange, parallel

from _cy_types cimport CSRMatrix, flt
from _cy_math  cimport flt_compare, qsort


def calculate_auc_from_rank(CSRMatrix ranks,
                             int[::1]  num_train_positives,
                             flt[::1]  rank_data,
                             flt[::1]  auc,
                             int       num_threads):
    """
    Convert per-user item ranks (from predict_ranks) to ROC-AUC scores.

    For each user the AUC is the probability that a randomly chosen
    positive example is ranked above a randomly chosen negative.
    A perfect model scores 1.0; random scoring gives 0.5.

    Parameters
    ----------
    ranks               : CSRMatrix  — sparse rank matrix (users × items)
    num_train_positives : int32 array [n_users]  — # train positives per user
                          (excluded from the negative count)
    rank_data           : float32 array  — the .data buffer of `ranks`
                          (modified in-place: sorted per-user row)
    auc                 : float32 array [n_users]  — output AUC per user
    num_threads         : int ≥ 1
    """
    fprintf(stderr,
            b"[DEBUG] calculate_auc_from_rank: n_users=%d num_threads=%d\n",
            ranks.rows, num_threads)

    cdef int i, user_id, row_start, row_stop
    cdef int num_negatives, num_positives
    cdef flt rank

    with nogil, parallel(num_threads=num_threads):
        for user_id in prange(ranks.rows, schedule='static'):
            row_start     = ranks.get_row_start(user_id)
            row_stop      = ranks.get_row_end(user_id)
            num_positives = row_stop - row_start
            num_negatives = (ranks.cols
                             - num_positives
                             - num_train_positives[user_id])

            # Degenerate case: only one class present → return 0.5
            if num_positives == 0 or num_negatives == ranks.cols:
                auc[user_id] = 0.5
                continue

            # Sort positive ranks ascending so we can correct for ties
            qsort(&rank_data[row_start],
                  num_positives,
                  sizeof(flt),
                  flt_compare)

            for i in range(num_positives):
                rank = ranks.data[row_start + i]

                # Subtract the i other positives that rank above this one.
                # Clamp to zero to avoid negative ranks.
                rank = rank - i
                if rank < 0:
                    rank = 0

                # P(positive ranked above random negative) for this item
                auc[user_id] += 1.0 - rank / num_negatives

            if num_positives != 0:
                auc[user_id] /= num_positives

    fprintf(stderr, b"[DEBUG] calculate_auc_from_rank: done\n")

Writing ./cy/_cy_evaluate.pyx


In [18]:
%%writefile ./cy/_cy_fit_bpr.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_fit_bpr.pyx
# BPR (Bayesian Personalised Ranking) training kernel for arycolbring.

import numpy as np
from libc.stdlib  cimport malloc, free
from libc.stdio   cimport fprintf, stderr
from cython.parallel cimport prange, parallel, threadid

from _cy_types          cimport CSRMatrix, FastAryColBring, flt
from _cy_math           cimport rand_r, in_positives, sigmoid
from _cy_representation cimport compute_representation, compute_prediction_from_repr
from _cy_update         cimport warp_update
from _cy_regularize     cimport regularize, locked_regularize, omp_lock_t, \
                                 omp_init_lock, omp_destroy_lock

cdef double MAX_REG_SCALE = 1000000.0


def fit_bpr(CSRMatrix item_features,
            CSRMatrix user_features,
            CSRMatrix interactions,
            int[::1] user_ids,
            int[::1] item_ids,
            flt[::1] Y,
            flt[::1] sample_weight,
            int[::1] shuffle_indices,
            FastAryColBring model,
            double learning_rate,
            double item_alpha,
            double user_alpha,
            int num_threads,
            random_state):
    """
    One epoch of BPR-loss collaborative filtering.

    For each positive, one hard negative is sampled and a pairwise
    gradient update (same kernel as WARP) is applied with a sigmoid loss.
    """
    fprintf(stderr,
            b"[DEBUG] fit_bpr: no_examples=%d num_threads=%d\n",
            Y.shape[0], num_threads)

    cdef int i, j, no_examples, user_id, positive_item_id, negative_item_id
    cdef int sampled, row
    cdef double positive_prediction, negative_prediction
    cdef flt weight
    cdef flt *user_repr
    cdef flt *pos_it_repr
    cdef flt *neg_it_repr
    cdef unsigned int[::1] random_states
    cdef omp_lock_t reg_lock

    random_states = (random_state
                     .randint(0, np.iinfo(np.int32).max, size=num_threads)
                     .astype(np.uint32))

    no_examples = Y.shape[0]

    if no_examples == 0:
        fprintf(stderr, b"[WARN] fit_bpr: no_examples=0, skipping\n")
        return

    omp_init_lock(&reg_lock)
    fprintf(stderr, b"[DEBUG] fit_bpr: OMP lock initialised\n")

    with nogil, parallel(num_threads=num_threads):
        user_repr   = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        pos_it_repr = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        neg_it_repr = <flt*>malloc(sizeof(flt) * (model.no_components + 1))

        if user_repr == NULL or pos_it_repr == NULL or neg_it_repr == NULL:
            fprintf(stderr, b"[ERROR] fit_bpr: malloc failed for thread buffer\n")
        else:
            for i in prange(no_examples, schedule='dynamic'):
                row = shuffle_indices[i]

                if not Y[row] > 0:
                    continue

                weight           = sample_weight[row]
                user_id          = user_ids[row]
                positive_item_id = item_ids[row]

                # Sample a negative item (not in user's positives)
                for j in range(no_examples):
                    negative_item_id = item_ids[
                        rand_r(&random_states[threadid()]) % no_examples]
                    if not in_positives(negative_item_id, user_id, interactions):
                        break

                compute_representation(user_features,
                                       model.user_features, model.user_biases,
                                       model, user_id, model.user_scale, user_repr)
                compute_representation(item_features,
                                       model.item_features, model.item_biases,
                                       model, positive_item_id, model.item_scale,
                                       pos_it_repr)
                compute_representation(item_features,
                                       model.item_features, model.item_biases,
                                       model, negative_item_id, model.item_scale,
                                       neg_it_repr)

                positive_prediction = compute_prediction_from_repr(
                    user_repr, pos_it_repr, model.no_components)
                negative_prediction = compute_prediction_from_repr(
                    user_repr, neg_it_repr, model.no_components)

                warp_update(
                    weight * (1.0 - sigmoid(<flt>(positive_prediction
                                                  - negative_prediction))),
                    item_features, user_features,
                    user_id, positive_item_id, negative_item_id,
                    user_repr, pos_it_repr, neg_it_repr,
                    model, item_alpha, user_alpha)

                if model.item_scale > MAX_REG_SCALE or model.user_scale > MAX_REG_SCALE:
                    locked_regularize(model, item_alpha, user_alpha,
                                      &reg_lock, MAX_REG_SCALE)

        free(user_repr)
        free(pos_it_repr)
        free(neg_it_repr)

    omp_destroy_lock(&reg_lock)

    regularize(model, item_alpha, user_alpha)

    fprintf(stderr, b"[DEBUG] fit_bpr: epoch complete\n")


Writing ./cy/_cy_fit_bpr.pyx


In [19]:
%%writefile ./cy/_cy_fit_logistic.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_fit_logistic.pyx
# Logistic-loss SGD training kernel for arycolbring.
# Parallelised with OpenMP prange; each thread owns its own malloc'd buffers.

import numpy as np
from libc.stdlib cimport malloc, free
from libc.stdio  cimport fprintf, stderr
from cython.parallel cimport prange, parallel

from _cy_types          cimport CSRMatrix, FastAryColBring, flt
from _cy_math           cimport sigmoid
from _cy_representation cimport compute_representation, compute_prediction_from_repr
from _cy_update         cimport update
from _cy_regularize     cimport regularize, locked_regularize, omp_lock_t, \
                                 omp_init_lock, omp_destroy_lock

cdef double MAX_REG_SCALE = 1000000.0


def fit_logistic(CSRMatrix item_features,
                 CSRMatrix user_features,
                 int[::1] user_ids,
                 int[::1] item_ids,
                 flt[::1] Y,
                 flt[::1] sample_weight,
                 int[::1] shuffle_indices,
                 FastAryColBring model,
                 double learning_rate,
                 double item_alpha,
                 double user_alpha,
                 int num_threads):
    """
    One epoch of logistic-loss collaborative filtering.

    Each worker thread allocates its own (user_repr, it_repr) buffers.
    A shared OMP lock serialises the periodic full-regularisation flush.
    """
    fprintf(stderr,
            b"[DEBUG] fit_logistic: no_examples=%d num_threads=%d\n",
            Y.shape[0], num_threads)

    cdef int i, row, user_id, item_id, no_examples
    cdef double prediction, loss
    cdef int y
    cdef flt y_row, weight
    cdef flt *user_repr
    cdef flt *it_repr
    cdef omp_lock_t reg_lock

    no_examples = Y.shape[0]

    if no_examples == 0:
        fprintf(stderr, b"[WARN] fit_logistic: no_examples=0, skipping\n")
        return

    omp_init_lock(&reg_lock)
    fprintf(stderr, b"[DEBUG] fit_logistic: OMP lock initialised\n")

    with nogil, parallel(num_threads=num_threads):
        user_repr = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        it_repr   = <flt*>malloc(sizeof(flt) * (model.no_components + 1))

        if user_repr == NULL or it_repr == NULL:
            fprintf(stderr, b"[ERROR] fit_logistic: malloc failed for thread buffer\n")
        else:
            for i in prange(no_examples, schedule='dynamic'):
                row     = shuffle_indices[i]
                user_id = user_ids[row]
                item_id = item_ids[row]
                weight  = sample_weight[row]

                compute_representation(user_features,
                                       model.user_features,
                                       model.user_biases,
                                       model, user_id,
                                       model.user_scale, user_repr)

                compute_representation(item_features,
                                       model.item_features,
                                       model.item_biases,
                                       model, item_id,
                                       model.item_scale, it_repr)

                prediction = sigmoid(
                    compute_prediction_from_repr(user_repr, it_repr,
                                                 model.no_components))

                y_row = Y[row]
                y = 1 if y_row > 0 else 0

                loss = weight * (prediction - y)

                update(loss, item_features, user_features,
                       user_id, item_id, user_repr, it_repr,
                       model, item_alpha, user_alpha)

                if model.item_scale > MAX_REG_SCALE or model.user_scale > MAX_REG_SCALE:
                    locked_regularize(model, item_alpha, user_alpha,
                                      &reg_lock, MAX_REG_SCALE)

        free(user_repr)
        free(it_repr)

    omp_destroy_lock(&reg_lock)

    regularize(model, item_alpha, user_alpha)

    fprintf(stderr, b"[DEBUG] fit_logistic: epoch complete\n")


Writing ./cy/_cy_fit_logistic.pyx


In [20]:
%%writefile ./cy/_cy_fit_warp.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_fit_warp.pyx
# WARP-loss kernel; no prange reduction on `sampled` / `sampled_local`.

import numpy as np
cimport numpy as np

from libc.stdlib  cimport malloc, free, rand
from libc.stdio   cimport fprintf, stderr

from _cy_types          cimport CSRMatrix, FastAryColBring, flt
from _cy_math           cimport in_positives
from _cy_representation cimport compute_representation, compute_prediction_from_repr
from _cy_update         cimport warp_update
from _cy_regularize     cimport regularize, locked_regularize, omp_lock_t, omp_init_lock, omp_destroy_lock

cdef extern from "math.h" nogil:
    double log(double)
    double floor(double)

cdef double MAX_REG_SCALE = 1000000.0


def fit_warp(
        CSRMatrix item_features,
        CSRMatrix user_features,
        CSRMatrix interactions,
        int[::1] user_ids,
        int[::1] item_ids,
        flt[::1] Y,
        flt[::1] sample_weight,
        int[::1] shuffle_indices,
        FastAryColBring model,
        double learning_rate,
        double item_alpha,
        double user_alpha,
        int num_threads,
        random_state):
    """
    One epoch of WARP-loss collaborative filtering.

    We keep the outer loop sequential; inner WARP loop is `nogil` but not `prange`.
    This avoids reduction‑variable errors on `sampled` and `sampled_local`.
    """
    fprintf(stderr,
            b"[DEBUG] fit_warp: no_examples=%d num_threads=%d\n",
            Y.shape[0], num_threads)

    cdef int i, no_examples, user_id, positive_item_id, negative_item_id
    cdef int sampled, sampled_local, row, local_seed_index
    cdef double positive_prediction, negative_prediction, loss
    cdef double MAX_LOSS = 10.0
    cdef flt weight
    cdef flt *user_repr
    cdef flt *pos_it_repr
    cdef flt *neg_it_repr
    cdef unsigned int[::1] random_states
    cdef omp_lock_t reg_lock

    random_states = (random_state
                     .randint(0, np.iinfo(np.int32).max, size=num_threads)
                     .astype(np.uintc))

    no_examples = Y.shape[0]

    if no_examples == 0:
        fprintf(stderr, b"[WARN] fit_warp: no_examples=0, skipping\n")
        return

    omp_init_lock(&reg_lock)
    fprintf(stderr, b"[DEBUG] fit_warp: OMP lock initialised\n")

    user_repr   = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
    pos_it_repr = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
    neg_it_repr = <flt*>malloc(sizeof(flt) * (model.no_components + 1))

    if user_repr == NULL or pos_it_repr == NULL or neg_it_repr == NULL:
        fprintf(stderr, b"[ERROR] fit_warp: malloc failed for thread buffer\n")
        if user_repr   != NULL: free(user_repr)
        if pos_it_repr != NULL: free(pos_it_repr)
        if neg_it_repr != NULL: free(neg_it_repr)
        omp_destroy_lock(&reg_lock)
        return

    # No `prange` here; keep outer loop sequential to avoid reduction warning.
    for i in range(no_examples):

        row              = shuffle_indices[i]
        user_id          = user_ids[row]
        positive_item_id = item_ids[row]

        if not Y[row] > 0:
            continue

        weight = sample_weight[row]

        compute_representation(user_features,
                               model.user_features, model.user_biases,
                               model, user_id, model.user_scale, user_repr)

        compute_representation(item_features,
                               model.item_features, model.item_biases,
                               model, positive_item_id, model.item_scale,
                               pos_it_repr)

        positive_prediction = compute_prediction_from_repr(
            user_repr, pos_it_repr, model.no_components)

        sampled = 0
        local_seed_index = i % num_threads
        while sampled < model.max_sampled:
            sampled += 1
            negative_item_id = rand() % item_features.rows

            compute_representation(item_features,
                                   model.item_features, model.item_biases,
                                   model, negative_item_id, model.item_scale,
                                   neg_it_repr)

            negative_prediction = compute_prediction_from_repr(
                user_repr, neg_it_repr, model.no_components)

            if negative_prediction > positive_prediction - 1:
                if in_positives(negative_item_id, user_id, interactions):
                    continue

                sampled_local = sampled
                loss = weight * log(
                    max(1.0,
                        floor((item_features.rows - 1) / <double>sampled_local)))

                if loss > MAX_LOSS:
                    loss = MAX_LOSS

                warp_update(loss, item_features, user_features,
                            user_id, positive_item_id, negative_item_id,
                            user_repr, pos_it_repr, neg_it_repr,
                            model, item_alpha, user_alpha)
                break

        if model.item_scale > MAX_REG_SCALE or model.user_scale > MAX_REG_SCALE:
            locked_regularize(model, item_alpha, user_alpha, &reg_lock, MAX_REG_SCALE)

    free(user_repr)
    free(pos_it_repr)
    free(neg_it_repr)

    omp_destroy_lock(&reg_lock)

    regularize(model, item_alpha, user_alpha)

    fprintf(stderr, b"[DEBUG] fit_warp: epoch complete\n")

Writing ./cy/_cy_fit_warp.pyx


In [21]:
%%writefile ./cy/_cy_fit_warp_kos.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_fit_warp_kos.pyx
# WARP k-OS kernel; no prange on outer loop; no reduction‑var error.

import numpy as np
cimport numpy as np

from libc.stdlib  cimport malloc, free, rand, qsort
from libc.stdio   cimport fprintf, stderr

from _cy_types          cimport CSRMatrix, FastAryColBring, flt
from _cy_math           cimport (sample_range, in_positives,
                                  int_min, Pair, reverse_pair_compare, qsort)
from _cy_representation cimport compute_representation, compute_prediction_from_repr
from _cy_update         cimport warp_update
from _cy_regularize     cimport regularize, locked_regularize, omp_lock_t, \
                               omp_init_lock, omp_destroy_lock

cdef extern from "math.h" nogil:
    double log(double)
    double floor(double)

cdef double MAX_REG_SCALE = 1000000.0


def fit_warp_kos(
        CSRMatrix item_features,
        CSRMatrix user_features,
        CSRMatrix data,
        int[::1] user_ids,
        int[::1] shuffle_indices,
        FastAryColBring model,
        double learning_rate,
        double item_alpha,
        double user_alpha,
        int k,
        int n,
        int num_threads,
        random_state):
    """
    One epoch of WARP-kOS.

    Outer loop is `range`, not `prange`, to avoid reduction‑variable warnings
    on `sampled` and `sampled_local`.
    """
    fprintf(stderr,
            b"[DEBUG] fit_warp_kos: no_examples=%d k=%d n=%d num_threads=%d\n",
            user_ids.shape[0], k, n, num_threads)

    cdef int i, j, no_examples, user_id, positive_item_id, negative_item_id
    cdef int sampled, sampled_local, row, sampled_positive_item_id
    cdef int user_pids_start, user_pids_stop, no_positives, idx
    cdef double positive_prediction, negative_prediction, sampled_positive_prediction
    cdef double loss, MAX_LOSS = 10.0
    cdef flt *user_repr
    cdef flt *pos_it_repr
    cdef flt *neg_it_repr
    cdef Pair *pos_pairs
    cdef unsigned int[::1] random_states
    cdef omp_lock_t reg_lock

    random_states = (random_state
                     .randint(0, np.iinfo(np.int32).max, size=num_threads)
                     .astype(np.uintc))

    no_examples = user_ids.shape[0]

    if no_examples == 0:
        fprintf(stderr, b"[WARN] fit_warp_kos: no_examples=0, skipping\n")
        return

    omp_init_lock(&reg_lock)
    fprintf(stderr, b"[DEBUG] fit_warp_kos: OMP lock initialised\n")

    user_repr   = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
    pos_it_repr = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
    neg_it_repr = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
    pos_pairs   = <Pair*>malloc(sizeof(Pair) * n)

    if user_repr == NULL or pos_it_repr == NULL or neg_it_repr == NULL or pos_pairs == NULL:
        fprintf(stderr, b"[ERROR] fit_warp_kos: malloc failed\n")
        if user_repr   != NULL: free(user_repr)
        if pos_it_repr != NULL: free(pos_it_repr)
        if neg_it_repr != NULL: free(neg_it_repr)
        if pos_pairs   != NULL: free(pos_pairs)
        omp_destroy_lock(&reg_lock)
        return

    # No `prange`; sequential outer loop avoids reduction warning.
    for i in range(no_examples):

        row     = shuffle_indices[i]
        user_id = user_ids[row]

        compute_representation(user_features,
                               model.user_features, model.user_biases,
                               model, user_id, model.user_scale, user_repr)

        user_pids_start = data.get_row_start(user_id)
        user_pids_stop  = data.get_row_end(user_id)

        if user_pids_stop == user_pids_start:
            continue

        no_positives = int_min(n, user_pids_stop - user_pids_start)
        for j in range(no_positives):
            idx = user_pids_start + rand() % (user_pids_stop - user_pids_start)
            sampled_positive_item_id = data.indices[idx]

            compute_representation(item_features,
                                   model.item_features, model.item_biases,
                                   model, sampled_positive_item_id,
                                   model.item_scale, pos_it_repr)

            sampled_positive_prediction = compute_prediction_from_repr(
                user_repr, pos_it_repr, model.no_components)

            pos_pairs[j].idx = sampled_positive_item_id
            pos_pairs[j].val = <flt>sampled_positive_prediction

        qsort(pos_pairs, no_positives, sizeof(Pair), reverse_pair_compare)

        positive_item_id    = pos_pairs[int_min(k, no_positives) - 1].idx
        positive_prediction = pos_pairs[int_min(k, no_positives) - 1].val

        compute_representation(item_features,
                               model.item_features, model.item_biases,
                               model, positive_item_id, model.item_scale,
                               pos_it_repr)

        sampled = 0
        while sampled < model.max_sampled:
            sampled += 1
            negative_item_id = rand() % item_features.rows

            compute_representation(item_features,
                                   model.item_features, model.item_biases,
                                   model, negative_item_id, model.item_scale,
                                   neg_it_repr)

            negative_prediction = compute_prediction_from_repr(
                user_repr, neg_it_repr, model.no_components)

            if negative_prediction > positive_prediction - 1:
                if in_positives(negative_item_id, user_id, data):
                    continue

                sampled_local = sampled
                loss = log(floor((item_features.rows - 1) / <double>sampled_local))
                if loss > MAX_LOSS:
                    loss = MAX_LOSS

                warp_update(loss, item_features, user_features,
                            user_id, positive_item_id, negative_item_id,
                            user_repr, pos_it_repr, neg_it_repr,
                            model, item_alpha, user_alpha)
                break

        if model.item_scale > MAX_REG_SCALE or model.user_scale > MAX_REG_SCALE:
            locked_regularize(model, item_alpha, user_alpha, &reg_lock, MAX_REG_SCALE)

    free(user_repr)
    free(pos_it_repr)
    free(neg_it_repr)
    free(pos_pairs)

    omp_destroy_lock(&reg_lock)

    regularize(model, item_alpha, user_alpha)

    fprintf(stderr, b"[DEBUG] fit_warp_kos: epoch complete\n")

Writing ./cy/_cy_fit_warp_kos.pyx


In [22]:
%%writefile ./cy/_cy_math.pxd

# _cy_math.pxd
# Inline math utilities for arycolbring Cython modules.
# All functions defined here are inlined into every module that cimports them.

from _cy_types cimport CSRMatrix, flt

# ── C standard library declarations ─────────────────────────────────────────

cdef extern from "math.h" nogil:
    double sqrt(double)
    double exp(double)
    double log(double)
    double floor(double)

cdef extern from "stdlib.h" nogil:
    void qsort(void *base, int nmemb, int size,
               int(*compar)(const void *, const void *)) nogil
    void* bsearch(const void *key, const void *base, int nmemb, int size,
                  int(*compar)(const void *, const void *)) nogil

# ── Pair struct (used by WARP-kOS) ──────────────────────────────────────────

cdef struct Pair:
    int idx
    flt val

# ── PRNG ────────────────────────────────────────────────────────────────────

cdef inline unsigned int temper(unsigned int x) nogil:
    cdef unsigned int and_1 = 0x9D2C5680
    cdef unsigned int and_2 = 0xEFC60000
    x = x ^ (x >> 11)
    x = x ^ (x << 7  & and_1)
    x = x ^ (x << 15 & and_2)
    x = x ^ (x >> 18)
    return x


cdef inline int rand_r(unsigned int *seed) nogil:
    seed[0] = seed[0] * 1103515245 + 12345
    return temper(seed[0]) / 2


cdef inline int sample_range(int min_val, int max_val, unsigned int *seed) nogil:
    cdef int val_range = max_val - min_val
    return min_val + (rand_r(seed) % val_range)

# ── Integer helpers ──────────────────────────────────────────────────────────

cdef inline int int_min(int x, int y) nogil:
    if x < y:
        return x
    return y


cdef inline int int_max(int x, int y) nogil:
    if x < y:
        return y
    return x

# ── Comparators for qsort / bsearch ─────────────────────────────────────────

cdef inline int int_compare(const void *a, const void *b) noexcept nogil:
    cdef int va = (<int*>a)[0]
    cdef int vb = (<int*>b)[0]
    if va > vb:
        return 1
    elif va < vb:
        return -1
    return 0


cdef inline int flt_compare(const void *a, const void *b) noexcept nogil:
    cdef flt va = (<flt*>a)[0]
    cdef flt vb = (<flt*>b)[0]
    if va > vb:
        return 1
    elif va < vb:
        return -1
    return 0


cdef inline int reverse_pair_compare(const void *a, const void *b) noexcept nogil:
    cdef flt diff = (<Pair*>a).val - (<Pair*>b).val
    if diff < 0:
        return 1
    return -1

# ── Sigmoid activation ───────────────────────────────────────────────────────

cdef inline flt sigmoid(flt v) nogil:
    return <flt>(1.0 / (1.0 + exp(-v)))

# ── Positives lookup (binary search in sorted CSR row) ───────────────────────

cdef inline int in_positives(int item_id,
                              int user_id,
                              CSRMatrix interactions) nogil:
    """
    Return 1 if item_id is in the sorted indices of user_id's CSR row.
    Uses bsearch for O(log k) lookup where k = nnz per row.
    """
    cdef int start_idx = interactions.get_row_start(user_id)
    cdef int stop_idx  = interactions.get_row_end(user_id)

    if bsearch(&item_id,
               &interactions.indices[start_idx],
               stop_idx - start_idx,
               sizeof(int),
               int_compare) == NULL:
        return 0
    return 1


Writing ./cy/_cy_math.pxd


In [23]:
%%writefile ./cy/_cy_math.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_math.pyx
# All math utilities are declared as inline cdef in _cy_math.pxd.
# This stub exists so Cython builds _cy_math as an extension module;
# the pxd body is inlined into every module that cimports from it.

# No standalone symbols needed here — everything is inline in the pxd.


Writing ./cy/_cy_math.pyx


In [24]:
%%writefile ./cy/_cy_predict.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_predict.pyx
# Prediction kernels for arycolbring: pointwise scores and item ranks.
# Both are parallelised with OpenMP prange.

from libc.stdlib  cimport malloc, free
from libc.stdio   cimport fprintf, stderr
from cython.parallel cimport prange, parallel

from _cy_types          cimport CSRMatrix, FastAryColBring, flt
from _cy_math           cimport in_positives, int_max
from _cy_representation cimport compute_representation, compute_prediction_from_repr


def predict_arycolbring(CSRMatrix item_features,
                        CSRMatrix user_features,
                        int[::1] user_ids,
                        int[::1] item_ids,
                        flt[::1] predictions,
                        FastAryColBring model,
                        int num_threads):
    """
    Compute pointwise prediction scores for (user_id, item_id) pairs.

    Parameters
    ----------
    item_features  : CSRMatrix  [n_item_features × n_item_feat_cols]
    user_features  : CSRMatrix  [n_user_features × n_user_feat_cols]
    user_ids       : int32 array, length = n_examples
    item_ids       : int32 array, length = n_examples
    predictions    : float32 array, length = n_examples  (output, written in-place)
    model          : FastAryColBring  holding all embedding state
    num_threads    : int  ≥ 1
    """
    fprintf(stderr,
            b"[DEBUG] predict_arycolbring: n_examples=%d num_threads=%d\n",
            predictions.shape[0], num_threads)

    cdef int i, no_examples
    cdef flt *user_repr
    cdef flt *it_repr

    no_examples = predictions.shape[0]

    if no_examples == 0:
        fprintf(stderr, b"[WARN] predict_arycolbring: no_examples=0, nothing to score\n")
        return

    with nogil, parallel(num_threads=num_threads):
        user_repr = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        it_repr   = <flt*>malloc(sizeof(flt) * (model.no_components + 1))

        if user_repr == NULL or it_repr == NULL:
            fprintf(stderr, b"[ERROR] predict_arycolbring: malloc failed\n")
        else:
            for i in prange(no_examples, schedule='static'):
                compute_representation(user_features,
                                       model.user_features, model.user_biases,
                                       model, user_ids[i],
                                       model.user_scale, user_repr)
                compute_representation(item_features,
                                       model.item_features, model.item_biases,
                                       model, item_ids[i],
                                       model.item_scale, it_repr)

                predictions[i] = compute_prediction_from_repr(
                    user_repr, it_repr, model.no_components)

        free(user_repr)
        free(it_repr)

    fprintf(stderr, b"[DEBUG] predict_arycolbring: scoring complete\n")


def predict_ranks(CSRMatrix item_features,
                  CSRMatrix user_features,
                  CSRMatrix test_interactions,
                  CSRMatrix train_interactions,
                  flt[::1]  ranks,
                  FastAryColBring model,
                  int num_threads):
    """
    Compute the rank of every test-positive item for each user.

    For each user the routine:
      1. Scores the user's test-positive items.
      2. Scores ALL catalogue items (excluding train positives).
      3. Counts how many catalogue items outscore each test positive.
         That count is the item's rank (lower = better recommendation).

    Parameters
    ----------
    item_features     : CSRMatrix
    user_features     : CSRMatrix
    test_interactions : CSRMatrix   — positives to rank
    train_interactions: CSRMatrix   — positives to skip during ranking
    ranks             : float32 array matching test_interactions.data  (output)
    model             : FastAryColBring
    num_threads       : int ≥ 1
    """
    fprintf(stderr,
            b"[DEBUG] predict_ranks: n_users=%d num_threads=%d\n",
            test_interactions.rows, num_threads)

    cdef int i, j, user_id, item_id, predictions_size
    cdef int row_start, row_stop
    cdef flt *user_repr
    cdef flt *it_repr
    cdef int  *test_item_ids_buf
    cdef flt  *test_preds_buf
    cdef flt   prediction

    # Determine maximum row width (for buffer sizing)
    predictions_size = 0
    for user_id in range(test_interactions.rows):
        predictions_size = int_max(
            predictions_size,
            test_interactions.get_row_end(user_id)
            - test_interactions.get_row_start(user_id))

    fprintf(stderr,
            b"[DEBUG] predict_ranks: max_row_width=%d\n", predictions_size)

    if predictions_size == 0:
        fprintf(stderr, b"[WARN] predict_ranks: no test interactions found\n")
        return

    with nogil, parallel(num_threads=num_threads):
        user_repr       = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        it_repr         = <flt*>malloc(sizeof(flt) * (model.no_components + 1))
        test_item_ids_buf = <int*>malloc(sizeof(int) * predictions_size)
        test_preds_buf  = <flt*>malloc(sizeof(flt) * predictions_size)

        if (user_repr == NULL or it_repr == NULL
                or test_item_ids_buf == NULL or test_preds_buf == NULL):
            fprintf(stderr, b"[ERROR] predict_ranks: malloc failed\n")
        else:
            for user_id in prange(test_interactions.rows, schedule='dynamic'):
                row_start = test_interactions.get_row_start(user_id)
                row_stop  = test_interactions.get_row_end(user_id)

                if row_stop == row_start:
                    continue  # no test interactions for this user

                # Step 1 – compute user representation
                compute_representation(user_features,
                                       model.user_features, model.user_biases,
                                       model, user_id,
                                       model.user_scale, user_repr)

                # Step 2 – score every test-positive item for this user
                for i in range(row_stop - row_start):
                    item_id = test_interactions.indices[row_start + i]
                    compute_representation(item_features,
                                           model.item_features, model.item_biases,
                                           model, item_id, model.item_scale, it_repr)
                    test_item_ids_buf[i] = item_id
                    test_preds_buf[i]    = compute_prediction_from_repr(
                        user_repr, it_repr, model.no_components)

                # Step 3 – count catalogue items that beat each test positive
                for item_id in range(test_interactions.cols):
                    if in_positives(item_id, user_id, train_interactions):
                        continue  # skip known train positives

                    compute_representation(item_features,
                                           model.item_features, model.item_biases,
                                           model, item_id, model.item_scale, it_repr)
                    prediction = compute_prediction_from_repr(
                        user_repr, it_repr, model.no_components)

                    for i in range(row_stop - row_start):
                        if item_id != test_item_ids_buf[i] and prediction >= test_preds_buf[i]:
                            ranks[row_start + i] += 1.0

        free(user_repr)
        free(it_repr)
        free(test_item_ids_buf)
        free(test_preds_buf)

    fprintf(stderr, b"[DEBUG] predict_ranks: ranking complete\n")


Writing ./cy/_cy_predict.pyx


In [25]:
%%writefile ./cy/_cy_regularize.pxd

# _cy_regularize.pxd
# Inline L2 regularisation helpers for arycolbring.
# regularize()        – flush accumulated lazy scale factor across all weights.
# locked_regularize() – thread-safe version using an OMP lock pointer.

from _cy_types cimport FastAryColBring, flt

cdef extern from "omp.h" nogil:
    ctypedef struct omp_lock_t:
        pass
    void omp_init_lock(omp_lock_t *) nogil
    void omp_destroy_lock(omp_lock_t *) nogil
    void omp_set_lock(omp_lock_t *) nogil
    void omp_unset_lock(omp_lock_t *) nogil


cdef inline void regularize(FastAryColBring model,
                             double item_alpha,
                             double user_alpha) nogil:
    """
    Flush the accumulated lazy-regularisation scale factors so that
    item_scale and user_scale are reset to 1.0.
    Every weight is divided by its current scale value.
    """
    cdef int i, j
    cdef int no_item_features = model.item_features.shape[0]
    cdef int no_user_features = model.user_features.shape[0]

    for i in range(no_item_features):
        for j in range(model.no_components):
            model.item_features[i, j] /= model.item_scale
        model.item_biases[i] /= model.item_scale

    for i in range(no_user_features):
        for j in range(model.no_components):
            model.user_features[i, j] /= model.user_scale
        model.user_biases[i] /= model.user_scale

    model.item_scale = 1.0
    model.user_scale = 1.0


cdef inline void locked_regularize(FastAryColBring model,
                                   double item_alpha,
                                   double user_alpha,
                                   omp_lock_t *lock,
                                   double MAX_REG_SCALE) nogil:
    """
    Thread-safe regularisation flush.
    Acquires *lock*, re-checks the threshold (another thread may have flushed
    already) and flushes if needed, then releases *lock*.
    """
    omp_set_lock(lock)
    if model.item_scale > MAX_REG_SCALE or model.user_scale > MAX_REG_SCALE:
        regularize(model, item_alpha, user_alpha)
    omp_unset_lock(lock)


Writing ./cy/_cy_regularize.pxd


In [26]:
%%writefile ./cy/_cy_regularize.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_regularize.pyx
# All regularise logic is inline in _cy_regularize.pxd.


Writing ./cy/_cy_regularize.pyx


In [27]:
%%writefile ./cy/_cy_representation.pxd

# _cy_representation.pxd
# Inline functions for computing latent representations and dot-product
# predictions for arycolbring collaborative filtering.

from _cy_types cimport CSRMatrix, FastAryColBring, flt


cdef inline void compute_representation(CSRMatrix features,
                                        flt[:, ::1] feature_embeddings,
                                        flt[::1]    feature_biases,
                                        FastAryColBring model,
                                        int  row_id,
                                        double scale,
                                        flt  *representation) nogil:
    """
    Accumulate the weighted embedding and bias for a given row (user or item).

    The output `representation` is a C array of length (no_components + 1):
      - indices 0 .. no_components-1 : latent factor values
      - index   no_components         : bias term
    """
    cdef int i, j, start_index, stop_index, feature
    cdef flt feature_weight

    start_index = features.get_row_start(row_id)
    stop_index  = features.get_row_end(row_id)

    # Zero-initialise output buffer
    for i in range(model.no_components + 1):
        representation[i] = 0.0

    for i in range(start_index, stop_index):
        feature        = features.indices[i]
        feature_weight = <flt>(features.data[i] * scale)

        for j in range(model.no_components):
            representation[j] += feature_weight * feature_embeddings[feature, j]

        # Bias sits at position no_components
        representation[model.no_components] += feature_weight * feature_biases[feature]


cdef inline flt compute_prediction_from_repr(flt *user_repr,
                                             flt *item_repr,
                                             int  no_components) nogil:
    """
    Compute the score for (user, item) as:
        user_bias + item_bias + dot(user_latent, item_latent)
    """
    cdef int i
    cdef flt result

    # Bias terms
    result = user_repr[no_components] + item_repr[no_components]

    # Dot product of latent factors
    for i in range(no_components):
        result += user_repr[i] * item_repr[i]

    return result


Writing ./cy/_cy_representation.pxd


In [28]:
%%writefile ./cy/_cy_representation.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_representation.pyx
# All logic is inline in _cy_representation.pxd.
# This stub exists to satisfy Cython's build system.


Writing ./cy/_cy_representation.pyx


In [29]:
%%writefile ./cy/_cy_types.pxd

# _cy_types.pxd
# Shared type declarations for arycolbring Cython modules.
# cimport this file in any .pyx that needs CSRMatrix or FastAryColBring.

ctypedef float flt


cdef class CSRMatrix:
    """
    Lightweight wrapper around a scipy CSR matrix for nogil access.
    Exposes indices/indptr/data as typed memory views.
    """
    cdef int[::1] indices
    cdef int[::1] indptr
    cdef flt[::1] data

    cdef int rows
    cdef int cols
    cdef int nnz

    cdef int get_row_start(self, int row) nogil
    cdef int get_row_end(self, int row) nogil


cdef class FastAryColBring:
    """
    Holds all model state (embeddings, gradients, momentum) for
    the arycolbring collaborative filtering model.
    All fields are typed memory views for direct C-level access.
    """
    cdef flt[:, ::1] item_features
    cdef flt[:, ::1] item_feature_gradients
    cdef flt[:, ::1] item_feature_momentum

    cdef flt[::1] item_biases
    cdef flt[::1] item_bias_gradients
    cdef flt[::1] item_bias_momentum

    cdef flt[:, ::1] user_features
    cdef flt[:, ::1] user_feature_gradients
    cdef flt[:, ::1] user_feature_momentum

    cdef flt[::1] user_biases
    cdef flt[::1] user_bias_gradients
    cdef flt[::1] user_bias_momentum

    cdef int no_components
    cdef int adadelta
    cdef flt learning_rate
    cdef flt rho
    cdef flt eps
    cdef int max_sampled

    cdef double item_scale
    cdef double user_scale


Writing ./cy/_cy_types.pxd


In [30]:
%%writefile ./cy/_cy_types.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_types.pyx
# Implementation of CSRMatrix and FastAryColBring cdef classes.

from libc.stdio cimport fprintf, stderr


cdef class CSRMatrix:
    """
    Thin wrapper over a scipy CSR sparse matrix.
    Provides nogil-safe row slicing via get_row_start / get_row_end.
    """

    def __init__(self, csr_matrix):
        fprintf(stderr, b"[DEBUG] CSRMatrix.__init__: wrapping sparse matrix\n")

        self.indices = csr_matrix.indices
        self.indptr  = csr_matrix.indptr
        self.data    = csr_matrix.data

        self.rows, self.cols = csr_matrix.shape
        self.nnz = len(self.data)

        fprintf(stderr,
                b"[DEBUG] CSRMatrix.__init__: rows=%d cols=%d nnz=%d\n",
                self.rows, self.cols, self.nnz)

    cdef int get_row_start(self, int row) nogil:
        return self.indptr[row]

    cdef int get_row_end(self, int row) nogil:
        return self.indptr[row + 1]


cdef class FastAryColBring:
    """
    Central model-state container.
    All embedding matrices and their optimiser accumulators live here.
    Passed by reference to every Cython kernel so they can mutate
    the arrays in-place without any Python overhead.
    """

    def __init__(self,
                 flt[:, ::1] item_features,
                 flt[:, ::1] item_feature_gradients,
                 flt[:, ::1] item_feature_momentum,
                 flt[::1]    item_biases,
                 flt[::1]    item_bias_gradients,
                 flt[::1]    item_bias_momentum,
                 flt[:, ::1] user_features,
                 flt[:, ::1] user_feature_gradients,
                 flt[:, ::1] user_feature_momentum,
                 flt[::1]    user_biases,
                 flt[::1]    user_bias_gradients,
                 flt[::1]    user_bias_momentum,
                 int         no_components,
                 int         adadelta,
                 flt         learning_rate,
                 flt         rho,
                 flt         epsilon,
                 int         max_sampled):

        fprintf(stderr,
                b"[DEBUG] FastAryColBring.__init__: no_components=%d adadelta=%d\n",
                no_components, adadelta)

        self.item_features           = item_features
        self.item_feature_gradients  = item_feature_gradients
        self.item_feature_momentum   = item_feature_momentum
        self.item_biases             = item_biases
        self.item_bias_gradients     = item_bias_gradients
        self.item_bias_momentum      = item_bias_momentum

        self.user_features           = user_features
        self.user_feature_gradients  = user_feature_gradients
        self.user_feature_momentum   = user_feature_momentum
        self.user_biases             = user_biases
        self.user_bias_gradients     = user_bias_gradients
        self.user_bias_momentum      = user_bias_momentum

        self.no_components  = no_components
        self.learning_rate  = learning_rate
        self.rho            = rho
        self.eps            = epsilon
        self.item_scale     = 1.0
        self.user_scale     = 1.0
        self.adadelta       = adadelta
        self.max_sampled    = max_sampled

        fprintf(stderr, b"[DEBUG] FastAryColBring.__init__: done\n")


Writing ./cy/_cy_types.pyx


In [31]:
%%writefile ./cy/_cy_update.pxd

# _cy_update.pxd
# Inline SGD update kernels for arycolbring.
# Supports both Adagrad and Adadelta learning-rate schedules.

from _cy_types cimport CSRMatrix, FastAryColBring, flt

cdef extern from "math.h" nogil:
    double sqrt(double)


# ── Bias update ──────────────────────────────────────────────────────────────

cdef inline double update_biases(CSRMatrix feature_indices,
                                 int start,
                                 int stop,
                                 flt[::1] biases,
                                 flt[::1] gradients,
                                 flt[::1] momentum,
                                 double gradient,
                                 int    adadelta,
                                 double learning_rate,
                                 double alpha,
                                 flt    rho,
                                 flt    eps) nogil:
    """
    Apply one SGD step on bias terms using Adagrad or Adadelta.
    Returns sum of per-feature local learning rates.
    """
    cdef int i, feature
    cdef double feature_weight, local_lr, update, sum_lr = 0.0

    if adadelta:
        for i in range(start, stop):
            feature        = feature_indices.indices[i]
            feature_weight = feature_indices.data[i]

            gradients[feature]  = (rho * gradients[feature]
                                   + (1.0 - rho) * (feature_weight * gradient) ** 2)
            local_lr            = (sqrt(momentum[feature] + eps)
                                   / sqrt(gradients[feature] + eps))
            update              = local_lr * gradient * feature_weight
            momentum[feature]   = (rho * momentum[feature]
                                   + (1.0 - rho) * update ** 2)
            biases[feature]    -= <flt>update
            # Lazy L2 regularisation: inflate scale, not every weight
            biases[feature]    *= <flt>(1.0 + alpha * local_lr)
            sum_lr             += local_lr
    else:
        for i in range(start, stop):
            feature        = feature_indices.indices[i]
            feature_weight = feature_indices.data[i]

            local_lr           = learning_rate / sqrt(gradients[feature])
            biases[feature]   -= <flt>(local_lr * feature_weight * gradient)
            gradients[feature] += <flt>((gradient * feature_weight) ** 2)
            biases[feature]   *= <flt>(1.0 + alpha * local_lr)
            sum_lr            += local_lr

    return sum_lr


# ── Embedding update ─────────────────────────────────────────────────────────

cdef inline double update_features(CSRMatrix  feature_indices,
                                   flt[:, ::1] features,
                                   flt[:, ::1] gradients,
                                   flt[:, ::1] momentum,
                                   int    component,
                                   int    start,
                                   int    stop,
                                   double gradient,
                                   int    adadelta,
                                   double learning_rate,
                                   double alpha,
                                   flt    rho,
                                   flt    eps) nogil:
    """
    Apply one SGD step for a single latent component across all active features.
    Returns sum of per-feature local learning rates.
    """
    cdef int i, feature
    cdef double feature_weight, local_lr, update, sum_lr = 0.0

    if adadelta:
        for i in range(start, stop):
            feature        = feature_indices.indices[i]
            feature_weight = feature_indices.data[i]

            gradients[feature, component]  = (
                rho * gradients[feature, component]
                + (1.0 - rho) * (feature_weight * gradient) ** 2)
            local_lr = (sqrt(momentum[feature, component] + eps)
                        / sqrt(gradients[feature, component] + eps))
            update   = local_lr * gradient * feature_weight
            momentum[feature, component] = (
                rho * momentum[feature, component] + (1.0 - rho) * update ** 2)
            features[feature, component] -= <flt>update
            features[feature, component] *= <flt>(1.0 + alpha * local_lr)
            sum_lr += local_lr
    else:
        for i in range(start, stop):
            feature        = feature_indices.indices[i]
            feature_weight = feature_indices.data[i]

            local_lr = learning_rate / sqrt(gradients[feature, component])
            features[feature, component]  -= <flt>(local_lr * feature_weight * gradient)
            gradients[feature, component] += <flt>((gradient * feature_weight) ** 2)
            features[feature, component]  *= <flt>(1.0 + alpha * local_lr)
            sum_lr += local_lr

    return sum_lr


# ── Logistic gradient step ────────────────────────────────────────────────────

cdef inline void update(double loss,
                        CSRMatrix item_features,
                        CSRMatrix user_features,
                        int user_id,
                        int item_id,
                        flt *user_repr,
                        flt *it_repr,
                        FastAryColBring model,
                        double item_alpha,
                        double user_alpha) nogil:
    """
    Apply the logistic gradient update for a single (user, item) pair.
    """
    cdef int i
    cdef int item_start = item_features.get_row_start(item_id)
    cdef int item_stop  = item_features.get_row_end(item_id)
    cdef int user_start = user_features.get_row_start(user_id)
    cdef int user_stop  = user_features.get_row_end(user_id)
    cdef double avg_lr  = 0.0
    cdef flt item_component, user_component

    avg_lr += update_biases(item_features, item_start, item_stop,
                            model.item_biases, model.item_bias_gradients,
                            model.item_bias_momentum,
                            loss, model.adadelta, model.learning_rate,
                            item_alpha, model.rho, model.eps)

    avg_lr += update_biases(user_features, user_start, user_stop,
                            model.user_biases, model.user_bias_gradients,
                            model.user_bias_momentum,
                            loss, model.adadelta, model.learning_rate,
                            user_alpha, model.rho, model.eps)

    for i in range(model.no_components):
        item_component = it_repr[i]
        user_component = user_repr[i]

        avg_lr += update_features(item_features, model.item_features,
                                  model.item_feature_gradients,
                                  model.item_feature_momentum,
                                  i, item_start, item_stop,
                                  loss * user_component,
                                  model.adadelta, model.learning_rate,
                                  item_alpha, model.rho, model.eps)

        avg_lr += update_features(user_features, model.user_features,
                                  model.user_feature_gradients,
                                  model.user_feature_momentum,
                                  i, user_start, user_stop,
                                  loss * item_component,
                                  model.adadelta, model.learning_rate,
                                  user_alpha, model.rho, model.eps)

    avg_lr /= ((model.no_components + 1) * (item_stop - item_start)
               + (model.no_components + 1) * (user_stop - user_start))

    model.item_scale *= 1.0 + item_alpha * avg_lr
    model.user_scale *= 1.0 + user_alpha * avg_lr


# ── WARP gradient step ────────────────────────────────────────────────────────

cdef inline void warp_update(double loss,
                             CSRMatrix item_features,
                             CSRMatrix user_features,
                             int user_id,
                             int positive_item_id,
                             int negative_item_id,
                             flt *user_repr,
                             flt *pos_it_repr,
                             flt *neg_it_repr,
                             FastAryColBring model,
                             double item_alpha,
                             double user_alpha) nogil:
    """
    Apply the WARP pairwise gradient update.
    Positive item receives a push-up; negative item receives a push-down.
    """
    cdef int i
    cdef int pos_start  = item_features.get_row_start(positive_item_id)
    cdef int pos_stop   = item_features.get_row_end(positive_item_id)
    cdef int neg_start  = item_features.get_row_start(negative_item_id)
    cdef int neg_stop   = item_features.get_row_end(negative_item_id)
    cdef int user_start = user_features.get_row_start(user_id)
    cdef int user_stop  = user_features.get_row_end(user_id)
    cdef double avg_lr  = 0.0
    cdef flt pos_comp, neg_comp, user_comp

    # Bias updates
    avg_lr += update_biases(item_features, pos_start, pos_stop,
                            model.item_biases, model.item_bias_gradients,
                            model.item_bias_momentum,
                            -loss, model.adadelta, model.learning_rate,
                            item_alpha, model.rho, model.eps)

    avg_lr += update_biases(item_features, neg_start, neg_stop,
                            model.item_biases, model.item_bias_gradients,
                            model.item_bias_momentum,
                            loss, model.adadelta, model.learning_rate,
                            item_alpha, model.rho, model.eps)

    avg_lr += update_biases(user_features, user_start, user_stop,
                            model.user_biases, model.user_bias_gradients,
                            model.user_bias_momentum,
                            loss, model.adadelta, model.learning_rate,
                            user_alpha, model.rho, model.eps)

    # Embedding updates
    for i in range(model.no_components):
        user_comp = user_repr[i]
        pos_comp  = pos_it_repr[i]
        neg_comp  = neg_it_repr[i]

        avg_lr += update_features(item_features, model.item_features,
                                  model.item_feature_gradients,
                                  model.item_feature_momentum,
                                  i, pos_start, pos_stop,
                                  -loss * user_comp,
                                  model.adadelta, model.learning_rate,
                                  item_alpha, model.rho, model.eps)

        avg_lr += update_features(item_features, model.item_features,
                                  model.item_feature_gradients,
                                  model.item_feature_momentum,
                                  i, neg_start, neg_stop,
                                  loss * user_comp,
                                  model.adadelta, model.learning_rate,
                                  item_alpha, model.rho, model.eps)

        avg_lr += update_features(user_features, model.user_features,
                                  model.user_feature_gradients,
                                  model.user_feature_momentum,
                                  i, user_start, user_stop,
                                  loss * (neg_comp - pos_comp),
                                  model.adadelta, model.learning_rate,
                                  user_alpha, model.rho, model.eps)

    avg_lr /= ((model.no_components + 1) * (user_stop - user_start)
               + (model.no_components + 1) * (pos_stop - pos_start)
               + (model.no_components + 1) * (neg_stop - neg_start))

    model.item_scale *= 1.0 + item_alpha * avg_lr
    model.user_scale *= 1.0 + user_alpha * avg_lr


Writing ./cy/_cy_update.pxd


In [32]:
%%writefile ./cy/_cy_update.pyx

#!python
#cython: boundscheck=False, wraparound=False, cdivision=True, initializedcheck=False
# _cy_update.pyx
# All update logic is inline in _cy_update.pxd.


Writing ./cy/_cy_update.pyx


In [33]:
%%writefile ./setup.py

# setup.py
"""
Build script for arycolbring Cython extensions.

Each Cython module is compiled as a separate shared library so they can
be debugged independently.  All modules are compiled with:
  - OpenMP for prange parallelism
  - aggressive optimisation flags
  - boundscheck / wraparound disabled at the compiler level

Usage
-----
    # Compile in-place (development)
    python setup.py build_ext --inplace

    # Install
    pip install .

Requirements
------------
  - Cython >= 0.29 or 3.x
  - A C compiler with OpenMP support:
      Linux  : GCC  (gcc -fopenmp)
      macOS  : clang via Homebrew  (brew install llvm; CC=clang-XX)
      Windows: MSVC (/openmp)
"""

import os
import sys
import numpy as np
from setuptools import setup, find_packages, Extension
from Cython.Build import cythonize

# ── compiler flags ────────────────────────────────────────────────────────────
if sys.platform == "win32":
    extra_compile_args = ["/O2", "/openmp"]
    extra_link_args    = []
    omp_lib            = []
elif sys.platform == "darwin":
    # Homebrew LLVM clang supports -fopenmp; Apple clang does not.
    extra_compile_args = ["-O3", "-fopenmp", "-march=native"]
    extra_link_args    = ["-fopenmp"]
    omp_lib            = ["omp"]
else:  # Linux
    extra_compile_args = [
        "-O3",
        "-fopenmp",
        "-march=native",
        "-ffast-math",
    ]
    extra_link_args = ["-fopenmp"]
    omp_lib         = []

include_dirs = [np.get_include()]
cy_dir       = os.path.realpath("./cy")


def cy_ext(name: str, sources=None) -> Extension:
    """
    Create an Extension for a Cython module inside ``arycolbring/cy/``.

    Parameters
    ----------
    name    : dotted module name, e.g. ``"arycolbring.cy._cy_types"``
    sources : list of .pyx paths; defaults to the single .pyx matching *name*
    """
    if sources is None:
        module_file = name.split(".")[-1] + ".pyx"
        sources     = [os.path.join(cy_dir, module_file)]
    return Extension(
        name,
        sources=sources,
        include_dirs=include_dirs,
        extra_compile_args=extra_compile_args,
        extra_link_args=extra_link_args,
        libraries=omp_lib,
        language="c",
    )


# ── extension modules ─────────────────────────────────────────────────────────
# Order matters: shared types first so the .pxd is on the include path
# when downstream modules are compiled.

extensions = [
    cy_ext("cy._cy_types"),
    cy_ext("cy._cy_math"),
    cy_ext("cy._cy_representation"),
    cy_ext("cy._cy_update"),
    cy_ext("cy._cy_regularize"),
    cy_ext("cy._cy_fit_logistic"),
    cy_ext("cy._cy_fit_warp"),
    cy_ext("cy._cy_fit_bpr"),
    cy_ext("cy._cy_fit_warp_kos"),
    cy_ext("cy._cy_predict"),
    cy_ext("cy._cy_evaluate"),
]

# ── Cython compiler directives ────────────────────────────────────────────────
compiler_directives = {
    "boundscheck":      False,
    "wraparound":       False,
    "cdivision":        True,
    "initializedcheck": False,
    "nonecheck":        False,
    "embedsignature":   True,    # Allows introspection of compiled functions
    "language_level":   "3",
}

# ── setup ─────────────────────────────────────────────────────────────────────
setup(
    name="arycolbring",
    version="0.1.0",
    description=(
        "Ultra-optimised user-to-item collaborative filtering "
        "with Cython + OpenMP kernels"
    ),
    author="aryanto",
    python_requires=">=3.8",
    packages=find_packages(exclude=["tests*"]),
    package_data={
        "arycolbring":    ["config.ini"],
        "arycolbring.cy": ["*.pxd", "*.pyx"],
    },
    ext_modules=cythonize(
        extensions,
        compiler_directives=compiler_directives,
        annotate=True,      # produces _cy_*.html annotation files for profiling
        nthreads=4,         # parallel Cython transpilation (not OpenMP)
    ),
    include_dirs=include_dirs,
    install_requires=[
        "numpy>=1.21",
        "scipy>=1.7",
        "pandas>=1.3",
        "duckdb>=0.8",
        "tqdm>=4.60",
        "joblib>=1.1",
        "cython>=0.29",
        "seaborn>=0.12",
    ],
    zip_safe=False,
)


Writing ./setup.py


## Compiled them all!

In [34]:
!python setup.py build_ext --inplace -q

Compiling /kaggle/working/cy/_cy_types.pyx because it changed.
Compiling /kaggle/working/cy/_cy_math.pyx because it changed.
Compiling /kaggle/working/cy/_cy_representation.pyx because it changed.
Compiling /kaggle/working/cy/_cy_update.pyx because it changed.
Compiling /kaggle/working/cy/_cy_regularize.pyx because it changed.
Compiling /kaggle/working/cy/_cy_fit_logistic.pyx because it changed.
Compiling /kaggle/working/cy/_cy_fit_warp.pyx because it changed.
Compiling /kaggle/working/cy/_cy_fit_bpr.pyx because it changed.
Compiling /kaggle/working/cy/_cy_fit_warp_kos.pyx because it changed.
Compiling /kaggle/working/cy/_cy_predict.pyx because it changed.
Compiling /kaggle/working/cy/_cy_evaluate.pyx because it changed.
[ 1/11] Cythonizing /kaggle/working/cy/_cy_evaluate.pyx
[ 2/11] Cythonizing /kaggle/working/cy/_cy_fit_bpr.pyx
[ 4/11] Cythonizing /kaggle/working/cy/_cy_fit_warp.pyx
[ 3/11] Cythonizing /kaggle/working/cy/_cy_fit_logistic.pyx
performance hint: cy/_cy_math.pxd:104:51: 

## Set Config

In [35]:
%%writefile config.ini

[tqdm]
colour = #05ad46
ncols = 88

[model]
dtype = float32
no_components = 10
loss = warp
learning_schedule = adagrad
epochs = 10
num_threads = 4

[duckdb]
threads = 4

[logging]
level = INFO


Writing config.ini


In [36]:
workingdir = os.path.realpath('./')
print(workingdir)

/kaggle/working


## Cross Validation

In [37]:
from __future__ import annotations

import gc
import logging
import configparser
import os
from typing import Optional, Tuple

import numpy as np
import scipy.sparse as sp
from tqdm import tqdm

logger = logging.getLogger(__name__)

_cfg = configparser.ConfigParser()
_cfg.read(os.path.join(os.path.dirname(workingdir), "config.ini"))

TQDM_COLOUR = _cfg.get("tqdm", "colour", fallback="#05ad46")
TQDM_NCOLS  = _cfg.getint("tqdm", "ncols", fallback=80)


def _shuffle(
    uids: np.ndarray,
    iids: np.ndarray,
    data: np.ndarray,
    random_state: np.random.RandomState,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """In-place shuffle of COO triplets via a permutation index array."""
    shuffle_indices = np.arange(len(uids), dtype=np.int32)
    random_state.shuffle(shuffle_indices)
    return uids[shuffle_indices], iids[shuffle_indices], data[shuffle_indices]


def random_train_test_split(
    interactions:    sp.spmatrix,
    test_percentage: float = 0.2,
    random_state:    Optional[int | np.random.RandomState] = None,
) -> Tuple[sp.coo_matrix, sp.coo_matrix]:
    """
    Randomly split an interaction matrix into train and test sets.

    Non-zero entries are shuffled and the last ``test_percentage`` fraction
    is allocated to the test set.  The train set keeps the remaining
    interactions.  Both matrices preserve the original shape so that
    user/item index spaces are identical.

    Parameters
    ----------
    interactions    : scipy sparse matrix of any format (auto-converted to COO)
    test_percentage : fraction of interactions to use as test  (default 0.20)
    random_state    : int seed or ``np.random.RandomState`` instance

    Returns
    -------
    (train, test) : pair of scipy.sparse.coo_matrix

    Raises
    ------
    TypeError   – if interactions is not a scipy sparse matrix
    ValueError  – if test_percentage is not in (0, 1)
    RuntimeError – if the matrix has no non-zero entries
    """
    logger.debug(
        "random_train_test_split: shape=%s test_pct=%.2f",
        interactions.shape if hasattr(interactions, "shape") else "?",
        test_percentage,
    )

    if not sp.issparse(interactions):
        raise TypeError(
            "interactions must be a scipy sparse matrix, "
            f"got {type(interactions).__name__}."
        )
    if not (0 < test_percentage < 1):
        raise ValueError(
            f"test_percentage must be in (0, 1), got {test_percentage}."
        )
    if interactions.nnz == 0:
        raise RuntimeError(
            "interactions has no non-zero entries — cannot split an empty matrix."
        )

    if not isinstance(random_state, np.random.RandomState):
        random_state = np.random.RandomState(seed=random_state)

    with tqdm(total=4, desc="Splitting interactions",
              colour=TQDM_COLOUR, ncols=TQDM_NCOLS) as pbar:

        pbar.set_postfix_str("converting to COO")
        interactions = interactions.tocoo()
        shape = interactions.shape
        uids  = np.array(interactions.row,  dtype=np.int32)
        iids  = np.array(interactions.col,  dtype=np.int32)
        data  = np.array(interactions.data, dtype=interactions.dtype)
        pbar.update(1)

        pbar.set_postfix_str("shuffling entries")
        uids, iids, data = _shuffle(uids, iids, data, random_state)
        pbar.update(1)

        pbar.set_postfix_str("computing cutoff")
        cutoff    = int((1.0 - test_percentage) * len(uids))
        train_idx = slice(None, cutoff)
        test_idx  = slice(cutoff, None)
        logger.debug(
            "random_train_test_split: total=%d train=%d test=%d",
            len(uids), cutoff, len(uids) - cutoff,
        )
        pbar.update(1)

        pbar.set_postfix_str("building sparse matrices")
        train = sp.coo_matrix(
            (data[train_idx], (uids[train_idx], iids[train_idx])),
            shape=shape,
            dtype=interactions.dtype,
        )
        test = sp.coo_matrix(
            (data[test_idx], (uids[test_idx], iids[test_idx])),
            shape=shape,
            dtype=interactions.dtype,
        )
        pbar.update(1)

    del uids, iids, data
    gc.collect()
    logger.debug("random_train_test_split: done  train.nnz=%d test.nnz=%d",
                 train.nnz, test.nnz)

    return train, test


def user_based_train_test_split(
    interactions:    sp.spmatrix,
    test_percentage: float = 0.2,
    random_state:    Optional[int | np.random.RandomState] = None,
) -> Tuple[sp.coo_matrix, sp.coo_matrix]:
    """
    Split interactions such that each user's interactions are split
    independently.  Ensures every user appears in both train and test
    (as long as they have ≥ 2 interactions).

    Users with only one interaction are placed entirely in train.

    Parameters
    ----------
    interactions    : scipy sparse matrix
    test_percentage : fraction of each user's interactions for test
    random_state    : int seed or ``np.random.RandomState``

    Returns
    -------
    (train, test) : pair of scipy.sparse.coo_matrix
    """
    logger.debug(
        "user_based_train_test_split: shape=%s test_pct=%.2f",
        interactions.shape, test_percentage,
    )

    if not sp.issparse(interactions):
        raise TypeError(
            "interactions must be a scipy sparse matrix, "
            f"got {type(interactions).__name__}."
        )
    if not (0 < test_percentage < 1):
        raise ValueError(
            f"test_percentage must be in (0, 1), got {test_percentage}."
        )

    if not isinstance(random_state, np.random.RandomState):
        random_state = np.random.RandomState(seed=random_state)

    csr = interactions.tocsr()
    n_users = csr.shape[0]

    train_rows, train_cols, train_data = [], [], []
    test_rows,  test_cols,  test_data  = [], [], []

    with tqdm(total=n_users, desc="User-based split",
              colour=TQDM_COLOUR, ncols=TQDM_NCOLS) as pbar:

        for user_id in range(n_users):
            start = csr.indptr[user_id]
            stop  = csr.indptr[user_id + 1]

            cols_u = csr.indices[start:stop]
            data_u = csr.data[start:stop]
            n_u    = len(cols_u)

            perm = random_state.permutation(n_u)
            n_test = max(0, int(np.floor(n_u * test_percentage)))

            if n_u < 2:
                n_test = 0  # keep sole interaction in train

            test_idx  = perm[:n_test]
            train_idx = perm[n_test:]

            for idx in train_idx:
                train_rows.append(user_id)
                train_cols.append(cols_u[idx])
                train_data.append(data_u[idx])

            for idx in test_idx:
                test_rows.append(user_id)
                test_cols.append(cols_u[idx])
                test_data.append(data_u[idx])

            pbar.update(1)

    shape = interactions.shape
    dtype = interactions.dtype

    train = sp.coo_matrix(
        (np.array(train_data, dtype=dtype),
         (np.array(train_rows, dtype=np.int32),
          np.array(train_cols, dtype=np.int32))),
        shape=shape, dtype=dtype,
    )
    test = sp.coo_matrix(
        (np.array(test_data, dtype=dtype),
         (np.array(test_rows, dtype=np.int32),
          np.array(test_cols, dtype=np.int32))),
        shape=shape, dtype=dtype,
    )

    logger.debug("user_based_train_test_split: train.nnz=%d test.nnz=%d",
                 train.nnz, test.nnz)
    gc.collect()
    return train, test


## Data Utilization

In [38]:
from __future__ import annotations

import gc
import logging
import configparser
import os
from contextlib import contextmanager
from typing import Optional, Tuple, Union

import duckdb
import numpy as np
import pandas as pd
import scipy.sparse as sp
from tqdm import tqdm

# ── logging ──────────────────────────────────────────────────────────────────
logger = logging.getLogger(__name__)

# ── config ───────────────────────────────────────────────────────────────────
_cfg = configparser.ConfigParser()
_cfg.read(os.path.join(os.path.dirname(workingdir), "config.ini"))

TQDM_COLOUR   = _cfg.get("tqdm",   "colour",    fallback="#05ad46")
TQDM_NCOLS    = _cfg.getint("tqdm", "ncols",     fallback=80)
DEFAULT_DTYPE = _cfg.get("model",  "dtype",      fallback="float32")
DUCKDB_THREADS = _cfg.getint("duckdb", "threads", fallback=4)


# ── DuckDB connection context ─────────────────────────────────────────────────

@contextmanager
def duck_connection(threads: int = DUCKDB_THREADS):
    """
    Context manager that yields an in-process DuckDB connection
    and guarantees it is closed on exit.

    Usage::

        with duck_connection() as con:
            df = con.execute("SELECT * FROM my_table").df()
    """
    logger.debug("Opening DuckDB connection with %d threads", threads)
    con = duckdb.connect(":memory:")
    con.execute(f"PRAGMA threads={threads}")
    try:
        yield con
    finally:
        logger.debug("Closing DuckDB connection")
        con.close()


# ── Data loading helpers ──────────────────────────────────────────────────────

def load_interactions_from_df(
    df: pd.DataFrame,
    user_col:  str = "user_id",
    item_col:  str = "item_id",
    rating_col: Optional[str] = None,
    dtype: str = DEFAULT_DTYPE,
) -> Tuple[sp.coo_matrix, np.ndarray, np.ndarray]:
    """
    Convert a Pandas DataFrame of (user, item[, rating]) rows into a
    sparse COO interaction matrix.  Uses DuckDB for the encoding query
    so it stays efficient even for tens of millions of rows.

    Parameters
    ----------
    df         : DataFrame with at least ``user_col`` and ``item_col`` columns.
    user_col   : Name of the user identifier column.
    item_col   : Name of the item identifier column.
    rating_col : Name of an optional rating column.
                 If None, all interactions are set to 1.0.
    dtype      : NumPy dtype string for the sparse matrix data array.

    Returns
    -------
    interactions : scipy.sparse.coo_matrix  (n_users × n_items)
    user_ids     : np.ndarray  — mapping from integer index → original user id
    item_ids     : np.ndarray  — mapping from integer index → original item id
    """
    logger.debug(
        "load_interactions_from_df: shape=%s user_col=%s item_col=%s rating_col=%s",
        df.shape, user_col, item_col, rating_col,
    )

    required = {user_col, item_col}
    if rating_col:
        required.add(rating_col)
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"DataFrame is missing columns: {missing}")

    with tqdm(total=4, desc="Building interaction matrix",
              colour=TQDM_COLOUR, ncols=TQDM_NCOLS) as pbar:

        # Step 1 — register df in DuckDB and encode IDs
        with duck_connection() as con:
            pbar.set_postfix_str("registering dataframe")
            con.register("raw_df", df)

            rating_expr = (f'CAST("{rating_col}" AS DOUBLE)'
                           if rating_col else "1.0")

            encoded = con.execute(f"""
                SELECT
                    DENSE_RANK() OVER (ORDER BY "{user_col}") - 1 AS user_idx,
                    DENSE_RANK() OVER (ORDER BY "{item_col}") - 1 AS item_idx,
                    {rating_expr}                                  AS rating,
                    "{user_col}"                                   AS user_id,
                    "{item_col}"                                   AS item_id
                FROM raw_df
            """).df()
            pbar.update(1)

            # Step 2 — extract unique ID mappings
            pbar.set_postfix_str("extracting ID maps")
            user_map = con.execute(
                'SELECT DISTINCT user_idx, user_id FROM encoded ORDER BY user_idx'
            ).df()
            item_map = con.execute(
                'SELECT DISTINCT item_idx, item_id FROM encoded ORDER BY item_idx'
            ).df()
            pbar.update(1)

        # Step 3 — convert to numpy C arrays
        pbar.set_postfix_str("converting to C arrays")
        row_arr  = encoded["user_idx"].values.astype(np.int32)
        col_arr  = encoded["item_idx"].values.astype(np.int32)
        data_arr = encoded["rating"].values.astype(dtype)

        n_users = int(row_arr.max()) + 1
        n_items = int(col_arr.max()) + 1

        logger.debug(
            "load_interactions_from_df: n_users=%d n_items=%d nnz=%d",
            n_users, n_items, len(data_arr),
        )
        pbar.update(1)

        # Step 4 — build sparse matrix
        pbar.set_postfix_str("building COO sparse matrix")
        interactions = sp.coo_matrix(
            (data_arr, (row_arr, col_arr)),
            shape=(n_users, n_items),
            dtype=np.dtype(dtype),
        )
        user_ids = user_map["user_id"].values
        item_ids = item_map["item_id"].values
        pbar.update(1)

    del encoded, row_arr, col_arr, data_arr
    gc.collect()
    logger.debug("load_interactions_from_df: done")

    return interactions, user_ids, item_ids


def load_interactions_from_csv(
    path: str,
    user_col:   str = "user_id",
    item_col:   str = "item_id",
    rating_col: Optional[str] = None,
    dtype: str = DEFAULT_DTYPE,
) -> Tuple[sp.coo_matrix, np.ndarray, np.ndarray]:
    """
    Load interactions directly from a CSV file via DuckDB (zero-copy read).

    Parameters
    ----------
    path       : Absolute or relative path to the CSV file.
    user_col   : Column name for user identifiers.
    item_col   : Column name for item identifiers.
    rating_col : Optional rating column name.
    dtype      : NumPy dtype string.

    Returns
    -------
    Same tuple as ``load_interactions_from_df``.
    """
    logger.debug("load_interactions_from_csv: path=%s", path)

    if not os.path.isfile(path):
        raise FileNotFoundError(f"Interaction CSV not found: {path}")

    with tqdm(total=2, desc="Reading CSV", colour=TQDM_COLOUR,
              ncols=TQDM_NCOLS) as pbar:
        pbar.set_postfix_str("reading via DuckDB")

        with duck_connection() as con:
            rating_expr = (f'CAST("{rating_col}" AS DOUBLE)'
                           if rating_col else "1.0")
            df = con.execute(f"""
                SELECT "{user_col}", "{item_col}", {rating_expr} AS rating
                FROM read_csv_auto('{path}')
            """).df()
        pbar.update(1)

        pbar.set_postfix_str("encoding interactions")
        result = load_interactions_from_df(
            df,
            user_col=user_col,
            item_col=item_col,
            rating_col="rating",
            dtype=dtype,
        )
        pbar.update(1)

    del df
    gc.collect()
    return result


def describe_interactions(interactions: sp.spmatrix) -> pd.DataFrame:
    """
    Return a summary DataFrame of a sparse interaction matrix via DuckDB.

    Columns: n_users, n_items, nnz, density, avg_interactions_per_user,
             min_interactions_per_user, max_interactions_per_user.
    """
    logger.debug("describe_interactions: shape=%s", interactions.shape)

    mat = interactions.tocsr()
    row_counts = np.diff(mat.indptr).astype(np.float64)

    with duck_connection() as con:
        con.register("row_counts_arr",
                     pd.DataFrame({"nnz_per_user": row_counts}))
        stats = con.execute("""
            SELECT
                COUNT(*)                     AS n_users,
                AVG(nnz_per_user)            AS avg_interactions_per_user,
                MIN(nnz_per_user)            AS min_interactions_per_user,
                MAX(nnz_per_user)            AS max_interactions_per_user
            FROM row_counts_arr
        """).df()

    n_users, n_items = interactions.shape
    nnz     = interactions.nnz
    density = nnz / (n_users * n_items) if n_users * n_items > 0 else 0.0

    summary = pd.DataFrame({
        "n_users":                   [n_users],
        "n_items":                   [n_items],
        "nnz":                       [nnz],
        "density":                   [density],
        "avg_interactions_per_user": [stats["avg_interactions_per_user"].iloc[0]],
        "min_interactions_per_user": [stats["min_interactions_per_user"].iloc[0]],
        "max_interactions_per_user": [stats["max_interactions_per_user"].iloc[0]],
    })

    logger.debug("describe_interactions: %s", summary.to_dict(orient="records")[0])
    return summary


def validate_sparse_matrix(mat: sp.spmatrix, name: str = "matrix") -> None:
    """
    Raise informative errors if *mat* is not a valid finite sparse matrix.

    Raises
    ------
    TypeError         – if mat is not a scipy sparse matrix
    ValueError        – if mat contains NaN / Inf values
    RuntimeError      – if mat has zero rows or columns
    ReferenceError    – if mat.data is None or empty unexpectedly
    """
    logger.debug("validate_sparse_matrix: name=%s type=%s", name, type(mat))

    if not sp.issparse(mat):
        raise TypeError(
            f"Expected a scipy sparse matrix for '{name}', "
            f"got {type(mat).__name__}."
        )
    if mat.shape[0] == 0 or mat.shape[1] == 0:
        raise RuntimeError(
            f"Sparse matrix '{name}' has degenerate shape {mat.shape}."
        )
    if mat.nnz == 0:
        logger.warning("validate_sparse_matrix: '%s' has zero non-zero entries", name)
        return

    data = mat.data
    if data is None:
        raise ReferenceError(f"Sparse matrix '{name}' has None data array.")

    if not np.isfinite(data).all():
        raise ValueError(
            f"Sparse matrix '{name}' contains NaN or Inf values. "
            "Check your input data."
        )

    logger.debug("validate_sparse_matrix: '%s' OK — nnz=%d", name, mat.nnz)


## Evaluation

In [39]:
!mv ./cy ./cy_src
!mv ./build/lib.linux-x86_64-cpython-312/cy /kaggle/working/cy

In [40]:
%%writefile ./cy/__init__.py

from . import _cy_types
from . import _cy_evaluate

from ._cy_types         import CSRMatrix, FastAryColBring
from ._cy_fit_logistic  import fit_logistic
from ._cy_fit_warp      import fit_warp
from ._cy_fit_bpr       import fit_bpr
from ._cy_fit_warp_kos  import fit_warp_kos
from ._cy_predict       import predict_arycolbring
from ._cy_predict       import predict_ranks
from ._cy_evaluate      import calculate_auc_from_rank


Writing ./cy/__init__.py


In [41]:
from __future__ import annotations

import gc
import logging
import configparser
import os
from typing import Optional

import numpy as np
import scipy.sparse as sp
from tqdm import tqdm

import sys
#from .cy import CSRMatrix, calculate_auc_from_rank

sys.path.append(os.path.abspath("./cy"))

from cy._cy_types import CSRMatrix
from cy._cy_evaluate import calculate_auc_from_rank

logger = logging.getLogger(__name__)

_cfg = configparser.ConfigParser()
_cfg.read(os.path.join(os.path.dirname(workingdir), "config.ini"))

TQDM_COLOUR = _cfg.get("tqdm", "colour", fallback="#05ad46")
TQDM_NCOLS  = _cfg.getint("tqdm", "ncols", fallback=80)

__all__ = ["precision_at_k", "recall_at_k", "auc_score", "reciprocal_rank"]


def _get_ranks(
    model,
    test_interactions:   sp.spmatrix,
    train_interactions:  Optional[sp.spmatrix],
    user_features:       Optional[sp.spmatrix],
    item_features:       Optional[sp.spmatrix],
    num_threads:         int,
    check_intersections: bool,
    step_label:          str,
) -> sp.csr_matrix:
    """Shared rank-computation helper used by all metric functions."""
    logger.debug("_get_ranks: metric=%s num_threads=%d", step_label, num_threads)

    with tqdm(total=1, desc=f"Computing ranks ({step_label})",
              colour=TQDM_COLOUR, ncols=TQDM_NCOLS) as pbar:
        ranks = model.predict_rank(
            test_interactions,
            train_interactions=train_interactions,
            user_features=user_features,
            item_features=item_features,
            num_threads=num_threads,
            check_intersections=check_intersections,
        )
        pbar.update(1)

    return ranks


def precision_at_k(
    model,
    test_interactions:   sp.spmatrix,
    train_interactions:  Optional[sp.spmatrix] = None,
    k:                   int  = 10,
    user_features:       Optional[sp.spmatrix] = None,
    item_features:       Optional[sp.spmatrix] = None,
    preserve_rows:       bool = False,
    num_threads:         int  = 1,
    check_intersections: bool = True,
) -> np.ndarray:
    """
    Precision@k — fraction of the top-k recommendations that are true positives.

    Parameters
    ----------
    model               : fitted AryColBring instance
    test_interactions   : sparse [n_users × n_items]  — ground-truth positives
    train_interactions  : optional sparse — known positives to exclude from ranking
    k                   : cut-off rank
    user_features       : optional CSR [n_users × n_user_features]
    item_features       : optional CSR [n_items × n_item_features]
    preserve_rows       : if False, drop users with no test interactions
    num_threads         : OpenMP thread count
    check_intersections : raise if test/train overlap

    Returns
    -------
    np.ndarray float64, shape (n_active_users,) or (n_users,)
        Precision@k score for each user (0.0 – 1.0).
    """
    if num_threads < 1:
        raise ValueError("num_threads must be ≥ 1")
    if k < 1:
        raise ValueError("k must be ≥ 1")

    logger.debug("precision_at_k: k=%d num_threads=%d", k, num_threads)

    ranks = _get_ranks(model, test_interactions, train_interactions,
                       user_features, item_features,
                       num_threads, check_intersections, f"P@{k}")

    with tqdm(total=2, desc=f"Precision@{k}",
              colour=TQDM_COLOUR, ncols=TQDM_NCOLS) as pbar:
        pbar.set_postfix_str("masking top-k")
        ranks.data = np.less(ranks.data, k, ranks.data)
        precision  = np.squeeze(np.array(ranks.sum(axis=1))) / k
        pbar.update(1)

        pbar.set_postfix_str("filtering active users")
        if not preserve_rows:
            active = test_interactions.getnnz(axis=1) > 0
            precision = precision[active]
        pbar.update(1)

    gc.collect()
    logger.debug("precision_at_k: mean=%.4f", float(precision.mean()))
    return precision


def recall_at_k(
    model,
    test_interactions:   sp.spmatrix,
    train_interactions:  Optional[sp.spmatrix] = None,
    k:                   int  = 10,
    user_features:       Optional[sp.spmatrix] = None,
    item_features:       Optional[sp.spmatrix] = None,
    preserve_rows:       bool = False,
    num_threads:         int  = 1,
    check_intersections: bool = True,
) -> np.ndarray:
    """
    Recall@k — fraction of true positives recovered in the top-k results.

    A perfect score is 1.0.

    Parameters
    ----------
    Same as ``precision_at_k``.

    Returns
    -------
    np.ndarray float64, shape (n_active_users,) or (n_users,)
        Recall@k per user.
    """
    if num_threads < 1:
        raise ValueError("num_threads must be ≥ 1")
    if k < 1:
        raise ValueError("k must be ≥ 1")

    logger.debug("recall_at_k: k=%d num_threads=%d", k, num_threads)

    ranks = _get_ranks(model, test_interactions, train_interactions,
                       user_features, item_features,
                       num_threads, check_intersections, f"R@{k}")

    with tqdm(total=2, desc=f"Recall@{k}",
              colour=TQDM_COLOUR, ncols=TQDM_NCOLS) as pbar:
        pbar.set_postfix_str("masking top-k")
        ranks.data = np.less(ranks.data, k, ranks.data)
        retrieved  = np.squeeze(test_interactions.getnnz(axis=1)).astype(np.float64)
        hit        = np.squeeze(np.array(ranks.sum(axis=1), dtype=np.float64))
        pbar.update(1)

        pbar.set_postfix_str("filtering active users")
        if not preserve_rows:
            active    = test_interactions.getnnz(axis=1) > 0
            hit       = hit[active]
            retrieved = retrieved[active]
        pbar.update(1)

    # Guard against division by zero for users with 0 test positives
    with np.errstate(divide="ignore", invalid="ignore"):
        recall = np.where(retrieved > 0, hit / retrieved, 0.0)

    gc.collect()
    logger.debug("recall_at_k: mean=%.4f", float(recall.mean()))
    return recall


def auc_score(
    model,
    test_interactions:   sp.spmatrix,
    train_interactions:  Optional[sp.spmatrix] = None,
    user_features:       Optional[sp.spmatrix] = None,
    item_features:       Optional[sp.spmatrix] = None,
    preserve_rows:       bool = False,
    num_threads:         int  = 1,
    check_intersections: bool = True,
) -> np.ndarray:
    """
    ROC AUC — probability that a random positive ranks above a random negative.

    A perfect model scores 1.0; a random model scores 0.5.
    Users with no test positives or where all items are negative return 0.5.

    Parameters
    ----------
    Same as ``precision_at_k`` without the ``k`` argument.

    Returns
    -------
    np.ndarray float32, shape (n_active_users,) or (n_users,)
    """
    if num_threads < 1:
        raise ValueError("num_threads must be ≥ 1")

    logger.debug("auc_score: num_threads=%d", num_threads)

    ranks = _get_ranks(model, test_interactions, train_interactions,
                       user_features, item_features,
                       num_threads, check_intersections, "AUC")

    if not np.all(ranks.data >= 0):
        raise RuntimeError(
            "Rank data contains negative values — this should not happen. "
            "Please report this as a bug."
        )

    with tqdm(total=2, desc="AUC",
              colour=TQDM_COLOUR, ncols=TQDM_NCOLS) as pbar:
        pbar.set_postfix_str("preparing train positive counts")
        auc = np.zeros(ranks.shape[0], dtype=np.float32)

        if train_interactions is not None:
            num_train_positives = np.squeeze(
                np.array(train_interactions.getnnz(axis=1)).astype(np.int32)
            )
        else:
            num_train_positives = np.zeros(
                test_interactions.shape[0], dtype=np.int32
            )
        pbar.update(1)

        pbar.set_postfix_str("computing AUC via Cython kernel")
        calculate_auc_from_rank(
            CSRMatrix(ranks),
            num_train_positives,
            ranks.data,
            auc,
            num_threads,
        )
        pbar.update(1)

    if not preserve_rows:
        auc = auc[test_interactions.getnnz(axis=1) > 0]

    gc.collect()
    logger.debug("auc_score: mean=%.4f", float(auc.mean()))
    return auc


def reciprocal_rank(
    model,
    test_interactions:   sp.spmatrix,
    train_interactions:  Optional[sp.spmatrix] = None,
    user_features:       Optional[sp.spmatrix] = None,
    item_features:       Optional[sp.spmatrix] = None,
    preserve_rows:       bool = False,
    num_threads:         int  = 1,
    check_intersections: bool = True,
) -> np.ndarray:
    """
    Mean Reciprocal Rank — 1 / rank of the highest-ranked true positive.

    A perfect model scores 1.0 (true positive is rank-1).
    Users with no positives score 0.0.

    Parameters
    ----------
    Same as ``auc_score``.

    Returns
    -------
    np.ndarray float64, shape (n_active_users,) or (n_users,)
    """
    if num_threads < 1:
        raise ValueError("num_threads must be ≥ 1")

    logger.debug("reciprocal_rank: num_threads=%d", num_threads)

    ranks = _get_ranks(model, test_interactions, train_interactions,
                       user_features, item_features,
                       num_threads, check_intersections, "MRR")

    with tqdm(total=2, desc="Reciprocal Rank",
              colour=TQDM_COLOUR, ncols=TQDM_NCOLS) as pbar:
        pbar.set_postfix_str("converting ranks to reciprocals")
        ranks.data = 1.0 / (ranks.data + 1.0)
        pbar.update(1)

        pbar.set_postfix_str("taking per-user max")
        rr = np.squeeze(np.array(ranks.max(axis=1).todense()))
        if not preserve_rows:
            rr = rr[test_interactions.getnnz(axis=1) > 0]
        pbar.update(1)

    gc.collect()
    logger.debug("reciprocal_rank: mean=%.4f", float(rr.mean()))
    return rr


## Model API

In [42]:
from __future__ import annotations

import gc
import logging
import configparser
import os
from contextlib import contextmanager
from typing import Any, Dict, Optional, Union

import numpy as np
import scipy.sparse as sp
from joblib import Parallel, delayed
from tqdm import tqdm

from _cy_types import CSRMatrix, FastAryColBring
from _cy_fit_logistic import fit_logistic
from _cy_fit_warp import fit_warp
from _cy_fit_bpr import fit_bpr
from _cy_fit_warp_kos import fit_warp_kos
from _cy_predict import predict_arycolbring, predict_ranks

# ── logging ──────────────────────────────────────────────────────────────────
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.DEBUG,
                    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")

# ── config ───────────────────────────────────────────────────────────────────
_cfg = configparser.ConfigParser()
_cfg.read(os.path.join(workingdir, "config.ini"))

TQDM_COLOUR  = _cfg.get("tqdm",  "colour",  fallback="#05ad46")
TQDM_NCOLS   = _cfg.getint("tqdm", "ncols",  fallback=80)
CYTHON_DTYPE = np.float32


__all__ = ["AryColBring"]


class AryColBring:
    """
    Ultra-optimised user-to-item collaborative filtering model.

    Supports four training objectives:
      - ``"logistic"``  — point-wise logistic regression
      - ``"warp"``      — Weighted Approximate-Rank Pairwise loss
      - ``"bpr"``       — Bayesian Personalised Ranking
      - ``"warp-kos"``  — WARP k-th Order Statistic variant

    Two optimiser schedules:
      - ``"adagrad"``   — per-feature adaptive learning rate
      - ``"adadelta"``  — gradient-squared momentum adaptive LR

    Parameters
    ----------
    no_components   : int   — number of latent dimensions (default 10)
    k               : int   — k for warp-kos anchor selection (default 5)
    n               : int   — n samples for kos anchor pool (default 10)
    learning_schedule: str  — "adagrad" | "adadelta"
    loss            : str   — "logistic" | "warp" | "bpr" | "warp-kos"
    learning_rate   : float — base learning rate (default 0.05)
    rho             : float — Adadelta decay factor ∈ (0, 1) (default 0.95)
    epsilon         : float — numerical stability term (default 1e-6)
    item_alpha      : float — L2 regularisation weight for items (default 0.0)
    user_alpha      : float — L2 regularisation weight for users (default 0.0)
    max_sampled     : int   — max negative samples per positive (default 10)
    random_state    : int | np.random.RandomState | None
    """

    # ── constructor ───────────────────────────────────────────────────────────

    def __init__(
        self,
        no_components:    int   = 10,
        k:                int   = 5,
        n:                int   = 10,
        learning_schedule: str  = "adagrad",
        loss:             str   = "logistic",
        learning_rate:    float = 0.05,
        rho:              float = 0.95,
        epsilon:          float = 1e-6,
        item_alpha:       float = 0.0,
        user_alpha:       float = 0.0,
        max_sampled:      int   = 10,
        random_state             = None,
    ):
        logger.debug(
            "AryColBring.__init__: loss=%s schedule=%s no_components=%d",
            loss, learning_schedule, no_components,
        )

        # ── parameter validation ─────────────────────────────────────────────
        if item_alpha < 0.0:
            raise ValueError("item_alpha must be ≥ 0.0")
        if user_alpha < 0.0:
            raise ValueError("user_alpha must be ≥ 0.0")
        if no_components <= 0:
            raise ValueError("no_components must be > 0")
        if k <= 0:
            raise ValueError("k must be > 0")
        if n <= 0:
            raise ValueError("n must be > 0")
        if not (0 < rho < 1):
            raise ValueError("rho must be in (0, 1)")
        if epsilon < 0:
            raise ValueError("epsilon must be ≥ 0")
        if max_sampled < 1:
            raise ValueError("max_sampled must be a positive integer")
        if learning_schedule not in ("adagrad", "adadelta"):
            raise ValueError(
                f"learning_schedule must be 'adagrad' or 'adadelta', "
                f"got '{learning_schedule}'"
            )
        if loss not in ("logistic", "warp", "bpr", "warp-kos"):
            raise ValueError(
                f"loss must be one of 'logistic','warp','bpr','warp-kos', "
                f"got '{loss}'"
            )

        self._loss              = loss
        self._learning_schedule = learning_schedule
        self._no_components     = no_components
        self._learning_rate     = learning_rate
        self._k                 = int(k)
        self._n                 = int(n)
        self._rho               = rho
        self._epsilon           = epsilon
        self._max_sampled       = max_sampled
        self._item_alpha        = item_alpha
        self._user_alpha        = user_alpha

        if random_state is None:
            self._random_state = np.random.RandomState()
        elif isinstance(random_state, np.random.RandomState):
            self._random_state = random_state
        elif isinstance(random_state, int):
            self._random_state = np.random.RandomState(seed=random_state)
        else:
            raise TypeError(
                "random_state must be None, an int, or np.random.RandomState"
            )

        self._reset_state()

    # ── properties ────────────────────────────────────────────────────────────

    @property
    def loss(self) -> str:
        return self._loss

    @loss.setter
    def loss(self, value: str) -> None:
        if value not in ("logistic", "warp", "bpr", "warp-kos"):
            raise ValueError(f"Invalid loss '{value}'")
        self._loss = value

    @property
    def no_components(self) -> int:
        return self._no_components

    @no_components.setter
    def no_components(self, value: int) -> None:
        if value <= 0:
            raise ValueError("no_components must be > 0")
        self._no_components = int(value)

    @property
    def learning_rate(self) -> float:
        return self._learning_rate

    @learning_rate.setter
    def learning_rate(self, value: float) -> None:
        if value <= 0:
            raise ValueError("learning_rate must be > 0")
        self._learning_rate = float(value)

    @property
    def item_alpha(self) -> float:
        return self._item_alpha

    @item_alpha.setter
    def item_alpha(self, value: float) -> None:
        if value < 0:
            raise ValueError("item_alpha must be ≥ 0")
        self._item_alpha = float(value)

    @property
    def user_alpha(self) -> float:
        return self._user_alpha

    @user_alpha.setter
    def user_alpha(self, value: float) -> None:
        if value < 0:
            raise ValueError("user_alpha must be ≥ 0")
        self._user_alpha = float(value)

    @property
    def is_fitted(self) -> bool:
        """True once ``fit`` or ``fit_partial`` has been called."""
        return self.item_embeddings is not None

    @property
    def learning_schedule(self) -> str:
        return self._learning_schedule

    # ── internal state helpers ────────────────────────────────────────────────

    def _reset_state(self) -> None:
        logger.debug("AryColBring._reset_state: clearing all embeddings")
        self.item_embeddings         = None
        self.item_embedding_gradients = None
        self.item_embedding_momentum = None
        self.item_biases             = None
        self.item_bias_gradients     = None
        self.item_bias_momentum      = None

        self.user_embeddings         = None
        self.user_embedding_gradients = None
        self.user_embedding_momentum = None
        self.user_biases             = None
        self.user_bias_gradients     = None
        self.user_bias_momentum      = None

    def _check_initialized(self) -> None:
        attrs = [
            "item_embeddings", "item_embedding_gradients",
            "item_embedding_momentum", "item_biases",
            "item_bias_gradients", "item_bias_momentum",
            "user_embeddings", "user_embedding_gradients",
            "user_embedding_momentum", "user_biases",
            "user_bias_gradients", "user_bias_momentum",
        ]
        for attr in attrs:
            if getattr(self, attr) is None:
                raise RuntimeError(
                    "Model is not yet fitted.  Call fit() or fit_partial() first."
                )

    def _initialize(self,
                    no_components:     int,
                    no_item_features:  int,
                    no_user_features:  int) -> None:
        logger.debug(
            "_initialize: no_components=%d n_item_feat=%d n_user_feat=%d",
            no_components, no_item_features, no_user_features,
        )

        # Item embeddings ± 0.5 / no_components, uniform
        self.item_embeddings = (
            (self._random_state.rand(no_item_features, no_components) - 0.5)
            / no_components
        ).astype(CYTHON_DTYPE)
        self.item_embedding_gradients = np.zeros_like(self.item_embeddings)
        self.item_embedding_momentum  = np.zeros_like(self.item_embeddings)
        self.item_biases              = np.zeros(no_item_features, dtype=CYTHON_DTYPE)
        self.item_bias_gradients      = np.zeros_like(self.item_biases)
        self.item_bias_momentum       = np.zeros_like(self.item_biases)

        # User embeddings
        self.user_embeddings = (
            (self._random_state.rand(no_user_features, no_components) - 0.5)
            / no_components
        ).astype(CYTHON_DTYPE)
        self.user_embedding_gradients = np.zeros_like(self.user_embeddings)
        self.user_embedding_momentum  = np.zeros_like(self.user_embeddings)
        self.user_biases              = np.zeros(no_user_features, dtype=CYTHON_DTYPE)
        self.user_bias_gradients      = np.zeros_like(self.user_biases)
        self.user_bias_momentum       = np.zeros_like(self.user_biases)

        # Adagrad initialises accumulators to 1 to avoid divide-by-zero
        if self._learning_schedule == "adagrad":
            self.item_embedding_gradients += 1
            self.item_bias_gradients      += 1
            self.user_embedding_gradients += 1
            self.user_bias_gradients      += 1

    @staticmethod
    def _to_cython_dtype(mat: sp.spmatrix) -> sp.spmatrix:
        if mat.dtype != CYTHON_DTYPE:
            return mat.astype(CYTHON_DTYPE)
        return mat

    def _construct_feature_matrices(
        self,
        n_users: int,
        n_items: int,
        user_features: Optional[sp.spmatrix],
        item_features: Optional[sp.spmatrix],
    ):
        logger.debug(
            "_construct_feature_matrices: n_users=%d n_items=%d", n_users, n_items
        )

        if user_features is None:
            user_features = sp.identity(n_users, dtype=CYTHON_DTYPE, format="csr")
        else:
            user_features = user_features.tocsr()

        if item_features is None:
            item_features = sp.identity(n_items, dtype=CYTHON_DTYPE, format="csr")
        else:
            item_features = item_features.tocsr()

        if n_users > user_features.shape[0]:
            raise ValueError(
                f"n_users ({n_users}) exceeds user_features rows "
                f"({user_features.shape[0]})"
            )
        if n_items > item_features.shape[0]:
            raise ValueError(
                f"n_items ({n_items}) exceeds item_features rows "
                f"({item_features.shape[0]})"
            )

        if self.user_embeddings is not None:
            if self.user_embeddings.shape[0] < user_features.shape[1]:
                raise ValueError(
                    "user_features specifies more columns than embedding matrix rows: "
                    f"{user_features.shape[1]} vs {self.user_embeddings.shape[0]}"
                )
        if self.item_embeddings is not None:
            if self.item_embeddings.shape[0] < item_features.shape[1]:
                raise ValueError(
                    "item_features specifies more columns than embedding matrix rows: "
                    f"{item_features.shape[1]} vs {self.item_embeddings.shape[0]}"
                )

        user_features = self._to_cython_dtype(user_features)
        item_features = self._to_cython_dtype(item_features)
        return user_features, item_features

    def _get_positives_lookup_matrix(self, interactions: sp.coo_matrix) -> sp.csr_matrix:
        mat = interactions.tocsr()
        if not mat.has_sorted_indices:
            mat.sort_indices()
        return mat

    def _process_sample_weight(
        self,
        interactions:  sp.coo_matrix,
        sample_weight: Optional[sp.coo_matrix],
    ) -> np.ndarray:
        if sample_weight is not None:
            if self._loss == "warp-kos":
                raise NotImplementedError(
                    "Sample weights are not supported with warp-kos loss."
                )
            if not isinstance(sample_weight, sp.coo_matrix):
                raise TypeError("sample_weight must be a scipy COO matrix.")
            if sample_weight.shape != interactions.shape:
                raise ValueError(
                    "sample_weight and interactions must have the same shape."
                )
            if not (np.array_equal(interactions.row, sample_weight.row)
                    and np.array_equal(interactions.col, sample_weight.col)):
                raise ValueError(
                    "sample_weight and interactions entries must be in the same order."
                )
            data = sample_weight.data
            if data.dtype != CYTHON_DTYPE:
                data = data.astype(CYTHON_DTYPE)
            return data
        else:
            if np.array_equiv(interactions.data, 1.0):
                return interactions.data
            return np.ones_like(interactions.data, dtype=CYTHON_DTYPE)

    def _get_model_data(self) -> FastAryColBring:
        return FastAryColBring(
            self.item_embeddings,
            self.item_embedding_gradients,
            self.item_embedding_momentum,
            self.item_biases,
            self.item_bias_gradients,
            self.item_bias_momentum,
            self.user_embeddings,
            self.user_embedding_gradients,
            self.user_embedding_momentum,
            self.user_biases,
            self.user_bias_gradients,
            self.user_bias_momentum,
            self._no_components,
            int(self._learning_schedule == "adadelta"),
            self._learning_rate,
            self._rho,
            self._epsilon,
            self._max_sampled,
        )

    def _check_finite(self) -> None:
        for name, arr in [
            ("item_embeddings", self.item_embeddings),
            ("item_biases",     self.item_biases),
            ("user_embeddings", self.user_embeddings),
            ("user_biases",     self.user_biases),
        ]:
            if not np.isfinite(np.sum(arr)):
                raise ValueError(
                    f"Non-finite values detected in '{name}' after update. "
                    "Try reducing learning_rate or normalising input features."
                )

    def _check_input_finite(self, data: np.ndarray, name: str = "input") -> None:
        if not np.isfinite(np.sum(data)):
            raise ValueError(
                f"Non-finite values detected in '{name}'. "
                "Check your input for NaN or Inf."
            )

    def _check_test_train_intersections(
        self,
        test_mat:  sp.spmatrix,
        train_mat: Optional[sp.spmatrix],
    ) -> None:
        if train_mat is not None:
            n = test_mat.multiply(train_mat).nnz
            if n:
                raise ValueError(
                    f"test and train matrices share {n} interactions. "
                    "This will produce optimistic evaluation results. "
                    "Fix your data split before evaluating."
                )

    # ── static helpers ────────────────────────────────────────────────────────

    @staticmethod
    def _epoch_iterator(n_epochs: int, verbose: bool):
        if verbose:
            return tqdm(range(n_epochs), desc="Epoch",
                        colour=TQDM_COLOUR, ncols=TQDM_NCOLS)
        return range(n_epochs)

    # ── public training interface ─────────────────────────────────────────────

    def fit(
        self,
        interactions:   sp.spmatrix,
        user_features:  Optional[sp.spmatrix] = None,
        item_features:  Optional[sp.spmatrix] = None,
        sample_weight:  Optional[sp.coo_matrix] = None,
        epochs:         int  = 1,
        num_threads:    int  = 1,
        verbose:        bool = False,
    ) -> "AryColBring":
        """
        Fit the model from scratch (discards any previous state).

        Parameters
        ----------
        interactions  : sparse matrix [n_users × n_items]
                        Non-zero values are positive interactions.
        user_features : optional CSR matrix [n_users × n_user_features]
        item_features : optional CSR matrix [n_items × n_item_features]
        sample_weight : optional COO matrix matching interactions shape
        epochs        : number of training epochs
        num_threads   : OpenMP thread count (≥ 1)
        verbose       : show tqdm epoch bar

        Returns
        -------
        self
        """
        logger.debug("AryColBring.fit: epochs=%d num_threads=%d", epochs, num_threads)
        self._reset_state()
        return self.fit_partial(
            interactions,
            user_features=user_features,
            item_features=item_features,
            sample_weight=sample_weight,
            epochs=epochs,
            num_threads=num_threads,
            verbose=verbose,
        )

    def fit_partial(
        self,
        interactions:   sp.spmatrix,
        user_features:  Optional[sp.spmatrix] = None,
        item_features:  Optional[sp.spmatrix] = None,
        sample_weight:  Optional[sp.coo_matrix] = None,
        epochs:         int  = 1,
        num_threads:    int  = 1,
        verbose:        bool = False,
    ) -> "AryColBring":
        """
        Partially fit the model (preserves previous embedding state).
        Suitable for incremental / online learning.

        Parameters mirror ``fit``.
        """
        logger.debug(
            "AryColBring.fit_partial: loss=%s epochs=%d num_threads=%d",
            self._loss, epochs, num_threads,
        )

        if num_threads < 1:
            raise ValueError("num_threads must be ≥ 1")
        if epochs < 1:
            raise ValueError("epochs must be ≥ 1")

        # Convert to COO
        interactions = interactions.tocoo()
        if interactions.dtype != CYTHON_DTYPE:
            interactions.data = interactions.data.astype(CYTHON_DTYPE)

        validate_sparse_matrix(interactions, "interactions")

        sample_weight_data = self._process_sample_weight(interactions, sample_weight)

        n_users, n_items = interactions.shape
        user_features, item_features = self._construct_feature_matrices(
            n_users, n_items, user_features, item_features
        )

        # Validate all inputs for finiteness
        for arr, lbl in [
            (user_features.data,  "user_features"),
            (item_features.data,  "item_features"),
            (interactions.data,   "interactions"),
            (sample_weight_data,  "sample_weight"),
        ]:
            self._check_input_finite(arr, lbl)

        if self.item_embeddings is None:
            self._initialize(
                self._no_components,
                item_features.shape[1],
                user_features.shape[1],
            )

        if item_features.shape[1] != self.item_embeddings.shape[0]:
            raise ValueError(
                f"item_features has {item_features.shape[1]} columns but "
                f"embedding has {self.item_embeddings.shape[0]} rows."
            )
        if user_features.shape[1] != self.user_embeddings.shape[0]:
            raise ValueError(
                f"user_features has {user_features.shape[1]} columns but "
                f"embedding has {self.user_embeddings.shape[0]} rows."
            )

        # Positives lookup matrix (sorted indices required for bsearch)
        if self._loss in ("warp", "bpr", "warp-kos"):
            positives_lookup = CSRMatrix(
                self._get_positives_lookup_matrix(interactions)
            )

        shuffle_indices = np.arange(len(interactions.data), dtype=np.int32)

        with tqdm(total=epochs, desc="Training",
                  colour=TQDM_COLOUR, ncols=TQDM_NCOLS,
                  disable=not verbose) as pbar:

            for epoch in self._epoch_iterator(epochs, verbose=False):
                self._random_state.shuffle(shuffle_indices)
                model_data = self._get_model_data()

                self._run_epoch(
                    item_features, user_features,
                    interactions, sample_weight_data,
                    shuffle_indices, num_threads,
                    self._loss,
                    positives_lookup if self._loss in ("warp", "bpr", "warp-kos")
                    else None,
                    model_data,
                )

                self._check_finite()
                pbar.update(1)
                pbar.set_postfix({"epoch": epoch + 1, "loss": self._loss})
                logger.debug("fit_partial: epoch %d/%d done", epoch + 1, epochs)

        gc.collect()
        logger.debug("fit_partial: training complete")
        return self

    def _run_epoch(
        self,
        item_features:    sp.csr_matrix,
        user_features:    sp.csr_matrix,
        interactions:     sp.coo_matrix,
        sample_weight:    np.ndarray,
        shuffle_indices:  np.ndarray,
        num_threads:      int,
        loss:             str,
        positives_lookup,
        model_data:       FastAryColBring,
    ) -> None:
        """Dispatch to the correct Cython training kernel."""
        logger.debug("_run_epoch: loss=%s", loss)

        cy_item = CSRMatrix(item_features)
        cy_user = CSRMatrix(user_features)

        if loss == "warp":
            fit_warp(
                cy_item, cy_user, positives_lookup,
                interactions.row, interactions.col,
                interactions.data, sample_weight,
                shuffle_indices, model_data,
                self._learning_rate, self._item_alpha, self._user_alpha,
                num_threads, self._random_state,
            )
        elif loss == "bpr":
            fit_bpr(
                cy_item, cy_user, positives_lookup,
                interactions.row, interactions.col,
                interactions.data, sample_weight,
                shuffle_indices, model_data,
                self._learning_rate, self._item_alpha, self._user_alpha,
                num_threads, self._random_state,
            )
        elif loss == "warp-kos":
            fit_warp_kos(
                cy_item, cy_user, positives_lookup,
                interactions.row, shuffle_indices,
                model_data,
                self._learning_rate, self._item_alpha, self._user_alpha,
                self._k, self._n,
                num_threads, self._random_state,
            )
        else:  # logistic
            fit_logistic(
                cy_item, cy_user,
                interactions.row, interactions.col,
                interactions.data, sample_weight,
                shuffle_indices, model_data,
                self._learning_rate, self._item_alpha, self._user_alpha,
                num_threads,
            )

    # ── public inference interface ────────────────────────────────────────────

    def predict(
        self,
        user_ids:       Union[int, list, np.ndarray],
        item_ids:       Union[list, np.ndarray],
        item_features:  Optional[sp.spmatrix] = None,
        user_features:  Optional[sp.spmatrix] = None,
        num_threads:    int = 1,
    ) -> np.ndarray:
        """
        Compute prediction scores for (user_id, item_id) pairs.

        Parameters
        ----------
        user_ids     : int, list, or int32 ndarray
        item_ids     : list or int32 ndarray (same length as user_ids)
        item_features: optional CSR matrix [n_items × n_item_features]
        user_features: optional CSR matrix [n_users × n_user_features]
        num_threads  : OpenMP thread count

        Returns
        -------
        np.ndarray float32, shape (n_pairs,)
        """
        logger.debug("AryColBring.predict: num_threads=%d", num_threads)
        self._check_initialized()

        if isinstance(user_ids, int):
            user_ids = np.repeat(np.int32(user_ids), len(item_ids))

        # Convert Python list/tuple → numpy C array
        if isinstance(user_ids, (list, tuple)):
            user_ids = np.array(user_ids, dtype=np.int32)
        if isinstance(item_ids, (list, tuple)):
            item_ids = np.array(item_ids, dtype=np.int32)

        if user_ids.dtype != np.int32:
            user_ids = user_ids.astype(np.int32)
        if item_ids.dtype != np.int32:
            item_ids = item_ids.astype(np.int32)

        if len(user_ids) != len(item_ids):
            raise ValueError(
                f"user_ids length ({len(user_ids)}) != item_ids length ({len(item_ids)})"
            )
        if num_threads < 1:
            raise ValueError("num_threads must be ≥ 1")
        if user_ids.min() < 0 or item_ids.min() < 0:
            raise ValueError(
                "Negative user_id or item_id found.  Check for overflow or bad input."
            )

        n_users = int(user_ids.max()) + 1
        n_items = int(item_ids.max()) + 1

        user_features, item_features = self._construct_feature_matrices(
            n_users, n_items, user_features, item_features
        )

        predictions = np.empty(len(user_ids), dtype=np.float32)

        predict_arycolbring(
            CSRMatrix(item_features),
            CSRMatrix(user_features),
            user_ids,
            item_ids,
            predictions,
            self._get_model_data(),
            num_threads,
        )

        return predictions

    def predict_rank(
        self,
        test_interactions:   sp.spmatrix,
        train_interactions:  Optional[sp.spmatrix] = None,
        item_features:       Optional[sp.spmatrix] = None,
        user_features:       Optional[sp.spmatrix] = None,
        num_threads:         int  = 1,
        check_intersections: bool = True,
    ) -> sp.csr_matrix:
        """
        Compute item ranks for all test-positive interactions.

        Parameters
        ----------
        test_interactions  : sparse matrix [n_users × n_items]
        train_interactions : optional sparse matrix (positives to exclude)
        item_features      : optional CSR feature matrix
        user_features      : optional CSR feature matrix
        num_threads        : OpenMP thread count
        check_intersections: raise if test/train share interactions

        Returns
        -------
        scipy.sparse.csr_matrix  of same structure as test_interactions,
        where .data holds the rank of each test positive (0-based).
        """
        logger.debug("AryColBring.predict_rank: num_threads=%d", num_threads)
        self._check_initialized()

        if num_threads < 1:
            raise ValueError("num_threads must be ≥ 1")

        if check_intersections:
            self._check_test_train_intersections(test_interactions, train_interactions)

        n_users, n_items = test_interactions.shape
        user_features, item_features = self._construct_feature_matrices(
            n_users, n_items, user_features, item_features
        )

        if item_features.shape[1] != self.item_embeddings.shape[0]:
            raise ValueError("item_features column count mismatches embedding rows.")
        if user_features.shape[1] != self.user_embeddings.shape[0]:
            raise ValueError("user_features column count mismatches embedding rows.")

        test_interactions = test_interactions.tocsr()
        test_interactions = self._to_cython_dtype(test_interactions)

        if train_interactions is None:
            train_interactions = sp.csr_matrix(
                (n_users, n_items), dtype=CYTHON_DTYPE
            )
        else:
            train_interactions = train_interactions.tocsr()
            train_interactions = self._to_cython_dtype(train_interactions)

        ranks = sp.csr_matrix(
            (np.zeros_like(test_interactions.data),
             test_interactions.indices,
             test_interactions.indptr),
            shape=test_interactions.shape,
        )

        predict_ranks(
            CSRMatrix(item_features),
            CSRMatrix(user_features),
            CSRMatrix(test_interactions),
            CSRMatrix(train_interactions),
            ranks.data,
            self._get_model_data(),
            num_threads,
        )

        return ranks

    # ── representation getters ────────────────────────────────────────────────

    def get_item_representations(
        self,
        features: Optional[sp.spmatrix] = None,
    ):
        """
        Return (biases, embeddings) for items.

        If *features* is None, returns the raw embedding arrays directly.
        Otherwise projects through the feature matrix.
        """
        self._check_initialized()
        if features is None:
            return self.item_biases, self.item_embeddings
        features = sp.csr_matrix(features, dtype=CYTHON_DTYPE)
        return features * self.item_biases, features * self.item_embeddings

    def get_user_representations(
        self,
        features: Optional[sp.spmatrix] = None,
    ):
        """
        Return (biases, embeddings) for users.

        If *features* is None, returns the raw embedding arrays directly.
        Otherwise projects through the feature matrix.
        """
        self._check_initialized()
        if features is None:
            return self.user_biases, self.user_embeddings
        features = sp.csr_matrix(features, dtype=CYTHON_DTYPE)
        return features * self.user_biases, features * self.user_embeddings

    # ── sklearn-style get/set params ──────────────────────────────────────────

    def get_params(self, deep: bool = True) -> Dict[str, Any]:
        """Return a dict of constructor parameters (sklearn API compatible)."""
        return {
            "loss":              self._loss,
            "learning_schedule": self._learning_schedule,
            "no_components":     self._no_components,
            "learning_rate":     self._learning_rate,
            "k":                 self._k,
            "n":                 self._n,
            "rho":               self._rho,
            "epsilon":           self._epsilon,
            "max_sampled":       self._max_sampled,
            "item_alpha":        self._item_alpha,
            "user_alpha":        self._user_alpha,
            "random_state":      self._random_state,
        }

    def set_params(self, **params) -> "AryColBring":
        """Set parameters (sklearn API compatible)."""
        valid = set(self.get_params().keys())
        for key, value in params.items():
            if key not in valid:
                raise ValueError(
                    f"Invalid parameter '{key}' for AryColBring.  "
                    f"Valid params: {sorted(valid)}"
                )
            setattr(self, f"_{key}", value)
        return self

    # ── string representation ─────────────────────────────────────────────────

    def __repr__(self) -> str:
        return (
            f"AryColBring("
            f"loss='{self._loss}', "
            f"no_components={self._no_components}, "
            f"learning_schedule='{self._learning_schedule}', "
            f"learning_rate={self._learning_rate}, "
            f"item_alpha={self._item_alpha}, "
            f"user_alpha={self._user_alpha}"
            f")"
        )


# Training

## Module Loads

In [43]:
!pip install -q cloudpickle duckdb scipy

In [44]:
from __future__ import annotations
import gc
import os
import sys
import cloudpickle as pickle
import logging
import argparse
import time
import warnings
from contextlib import contextmanager
from pathlib import Path
from typing import Dict, Optional, Tuple
import duckdb
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import coo_matrix, csr_matrix
from tqdm import tqdm

In [45]:
warnings.filterwarnings("ignore", category=UserWarning)
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[
        logging.StreamHandler(sys.stderr),
        logging.FileHandler("hybrid_pipeline.log", mode="w"),
    ],
)
logger = logging.getLogger("hybrid_pipeline")

## Setting

In [46]:
# ── tqdm colour ───────────────────────────────────────────────────────────────
TQDM_COLOUR = "#05ad46"
TQDM_NCOLS  = 90

# ── default paths & hyper-params ─────────────────────────────────────────────
DEFAULT_TRAIN_PATH  = "trainfile.parquet"
DEFAULT_FPG_PATH    = "DataFPGpredict.parquet"
DEFAULT_ARTIFACTS   = "artifacts"

DEFAULT_FPG_WEIGHT  = 0.35   # weight for FPG-augmented signal in blended matrix
DEFAULT_CF_WEIGHT   = 0.65   # weight for base CF signal
DEFAULT_SUPPORT_CLIP_PCT = 0.99  # clip FPG support at this percentile (robustness)
DEFAULT_SCORE_CLIP_PCT   = 0.99  # clip decayed_interaction_score at this percentile

# User and item feature columns taken from trainfile
USER_FEAT_COLS = [
    "num_sessions",
    "total_user_interactions",
    "user_purchase_events",
    "user_total_units_purchased",
    "user_unique_products",
    "user_tenure_days",
    "primary_device_category",
    "primary_country",
    "primary_traffic_source",
]
ITEM_FEAT_COLS = [
    "department_id",
    "aisle_id",
    "brand",
    "category3",
    "total_item_interactions",
    "item_total_views",
    "item_total_add_to_carts",
    "item_total_checkouts",
    "item_total_purchases",
    "item_unique_users",
    "item_total_units_sold",
    "item_avg_price",
]

## Data Loading Colab

In [47]:
def load_data(
    train_df: pd.DataFrame,
    fpg_df:   pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load trainfile and FPG prediction parquet files via DuckDB.

    Returns
    -------
    train_df : DataFrame with all 42 GA4 columns
    fpg_df   : DataFrame with group_id / items / support
    """
    logger.info("Loading data files …")
    assert isinstance(train_df, pd.DataFrame), 'Wrong input of data trainning'
    assert isinstance(fpg_df, pd.DataFrame), 'Wrong input of data FPgrowth'
    train_df["user_id"]    = train_df["user_id"].astype(np.float64)
    train_df["product_id"] = train_df["product_id"].astype(str).str.strip()
    fpg_df["group_id"]     = fpg_df["group_id"].astype(np.float64)
    fpg_df["items"]        = fpg_df["items"].astype(str).str.strip()
    fpg_df["support"]      = fpg_df["support"].astype(np.float32)

    logger.info(
        "Loaded: train=%s  fpg=%s",
        train_df.shape, fpg_df.shape,
    )
    return train_df, fpg_df

In [48]:
train_df, fpg_df = load_data(mas_train, DataFPGpredict)

2026-04-24 05:00:06,798 [INFO] hybrid_pipeline: Loading data files …
2026-04-24 05:00:06,956 [INFO] hybrid_pipeline: Loaded: train=(1014426, 42)  fpg=(27290, 3)


## Encoding data

In [49]:
class IDEncoder:
    """
    Bidirectional mapping between raw IDs and contiguous integer indices.
    Thread-safe after construction (read-only after fit).
    """
    def __init__(self, name: str):
        self.name    = name
        self._id2idx: Dict = dict()
        self._idx2id: list = list()

    def fit(self, ids) -> "IDEncoder":
        #unique = sorted(set(ids), key=lambda x: str(x))
        unique = list(dict.fromkeys(ids))
        self._idx2id = unique
        self._id2idx = {v: i for i, v in enumerate(unique)}
        logger.debug("IDEncoder '%s': %d unique IDs", self.name, len(unique))
        return self

    def transform(self, ids) -> np.ndarray:
        out = np.array([self._id2idx.get(v, -1) for v in ids], dtype=np.int32)
        n_unknown = (out == -1).sum()
        if n_unknown:
            logger.warning(
                "IDEncoder '%s': %d unknown IDs mapped to -1",
                self.name, n_unknown,
            )
        return out

    def fit_transform(self, ids) -> np.ndarray:
        return self.fit(ids).transform(ids)

    def inverse(self, indices) -> list:
        return [self._idx2id[i] if 0 <= i < len(self._idx2id) else None
                for i in indices]

    def __len__(self) -> int:
        return len(self._idx2id)

    def __contains__(self, item) -> bool:
        return item in self._id2idx

## Feature Matrix builders

In [50]:
def _minmax_scale(arr: np.ndarray) -> np.ndarray:
    """Min-max scale to [0, 1], handling constant columns."""
    mn, mx = arr.min(axis=0), arr.max(axis=0)
    rng = mx - mn
    rng[rng == 0] = 1.0
    return (arr - mn) / rng

In [51]:
def build_user_feature_matrix(
    train_df:    pd.DataFrame,
    user_enc:    IDEncoder,
    feat_cols:   list = USER_FEAT_COLS,
) -> csr_matrix:
    '''
    Build a float32 CSR feature matrix [n_users × n_user_features].
    For near-stable columns (can vary slightly per user), the median
    across all rows for that user is taken.
    Uses DuckDB for the groupby aggregation.
    '''
    logger.debug("build_user_feature_matrix: %d features", len(feat_cols))
    with tqdm(total=3, desc="User features",
              colour=TQDM_COLOUR, ncols=TQDM_NCOLS) as pbar:

        # ── 1. Aggregation ─────────────────────────────────────
        pbar.set_postfix_str("aggregating via DuckDB")
        with duckdb_connection() as con:
            con.register_dataframe("tdf", train_df[["user_id"] + feat_cols])
            agg_expr = ", ".join(
                f'MEDIAN("{c}") AS "{c}"' for c in feat_cols
            )
            user_feats = con.query(f'SELECT user_id, {agg_expr} FROM tdf GROUP BY user_id')
        pbar.update(1)

        # ── 2. Encoding ────────────────────────────────────────
        pbar.set_postfix_str("encoding user IDs")
        user_feats["user_idx"] = user_enc.transform(
            user_feats["user_id"].values.astype(np.float64))
        user_feats = user_feats[user_feats["user_idx"] >= 0].copy()
        user_feats = user_feats.sort_values("user_idx").reset_index(drop=True)
        pbar.update(1)

        # ── 3. Extract values (TANPA dense matrix) ─────────────
        pbar.set_postfix_str("building CSR matrix")
        indices = user_feats["user_idx"].values.astype(np.int32)
        values  = user_feats[feat_cols].values.astype(np.float32)

        # scaling hanya ke data yang ada (pakai fungsi kamu)
        values = _minmax_scale(values).astype(np.float32)
        n_users = len(user_enc)
        n_feats = len(feat_cols)

        # build sparse langsung
        rows = np.repeat(indices, n_feats)
        cols = np.tile(np.arange(n_feats, dtype=np.int32), len(indices))
        data = values.flatten()
        mat = csr_matrix((data, (rows, cols)),
                          shape=(n_users, n_feats),
                          dtype=np.float32)
        pbar.update(1)

    logger.debug(
        "User feature matrix: %s  nnz=%d", mat.shape, mat.nnz)
    return mat

In [52]:
def build_item_feature_matrix(
    train_df:  pd.DataFrame,
    item_enc:  IDEncoder,
    feat_cols: list = ITEM_FEAT_COLS,
) -> csr_matrix:
    '''
    Build a float32 CSR feature matrix [n_items × n_item_features].
    Items from FPG that are not in trainfile get a zero feature vector
    (cold-start items — CF will still learn their embeddings via FPG interactions).
    '''
    logger.debug("build_item_feature_matrix: %d features", len(feat_cols))
    with tqdm(total=5, desc="Item features",
              colour=TQDM_COLOUR, ncols=TQDM_NCOLS) as pbar:

        # ── 1. Aggregation ─────────────────────────────────────
        pbar.set_postfix_str("aggregating via DuckDB")
        with duckdb_connection() as con:
            con.register_dataframe("tdf", train_df[["product_id"] + feat_cols])
            agg_expr = ", ".join(f'FIRST("{c}") AS "{c}"' for c in feat_cols)
            item_feats = con.query(
                f'SELECT product_id, {agg_expr} FROM tdf GROUP BY product_id')
        pbar.update(1)

        # ── 2. Encoding ────────────────────────────────────────
        pbar.set_postfix_str("encoding item IDs")
        item_feats["item_idx"] = item_enc.transform(
            item_feats["product_id"].values.astype(str))
        item_feats = item_feats[item_feats["item_idx"] >= 0].copy()
        item_feats = item_feats.sort_values("item_idx").reset_index(drop=True)
        pbar.update(1)

        # ── 3. Build sparse matrix (NO dense, NO loop) ─────────
        pbar.set_postfix_str("building CSR matrix")
        indices = item_feats["item_idx"].values.astype(np.int32)
        values  = item_feats[feat_cols].values.astype(np.float32)
        values = np.nan_to_num(values, nan=0.0)
        values = _minmax_scale(values).astype(np.float32)
        pbar.update(1)
        
        n_items = len(item_enc)
        n_feats = len(feat_cols)
        rows = np.repeat(indices, n_feats)
        cols = np.tile(np.arange(n_feats, dtype=np.int32), len(indices))
        data = values.flatten()
        pbar.update(1)
        
        mat = csr_matrix(
            (data, (rows, cols)),
            shape=(n_items, n_feats),
            dtype=np.float32,)
        pbar.update(1)

    logger.debug(
        "Item feature matrix: %s  nnz=%d", mat.shape, mat.nnz)
    return mat

## Interaction matrix builders

In [53]:
def build_base_interaction_matrix(
    train_df:          pd.DataFrame,
    user_enc:          IDEncoder,
    item_enc:          IDEncoder,
    score_col:         str   = "decayed_interaction_score",
    score_clip_pct:    float = DEFAULT_SCORE_CLIP_PCT,
) -> coo_matrix:
    """
    Build the base CF interaction matrix from trainfile.

    Weight = clipped + log1p-scaled decayed_interaction_score.
    Log-scaling compresses the heavy tail while preserving rank order.
    """
    logger.debug("build_base_interaction_matrix")

    with tqdm(total=4, desc="Base interaction matrix",
              colour=TQDM_COLOUR, ncols=TQDM_NCOLS) as pbar:

        pbar.set_postfix_str("clipping scores")
        clip_val = float(train_df[score_col].quantile(score_clip_pct))
        scores   = train_df[score_col].clip(upper=clip_val).values.astype(np.float32)
        # log1p compresses outliers, keeps positivity
        scores   = np.log1p(scores).astype(np.float32)
        pbar.update(1)

        pbar.set_postfix_str("encoding user IDs")
        user_idx = user_enc.transform(
            train_df["user_id"].values.astype(np.float64)
        )
        pbar.update(1)

        pbar.set_postfix_str("encoding item IDs")
        item_idx = item_enc.transform(
            train_df["product_id"].values.astype(str)
        )
        pbar.update(1)

        pbar.set_postfix_str("building COO matrix")
        # Filter out any unknowns (should be none after fitting encoders on same data)
        valid = (user_idx >= 0) & (item_idx >= 0)
        if not valid.all():
            logger.warning(
                "build_base_interaction_matrix: dropping %d rows with unknown IDs",
                (~valid).sum(),
            )

        mat = coo_matrix(
            (scores[valid], (user_idx[valid], item_idx[valid])),
            shape=(len(user_enc), len(item_enc)),
            dtype=np.float32,
        )
        mat = mat.tocsr()
        mat.sum_duplicates()
        mat = mat.tocoo()
        pbar.update(1)

    logger.info(
        "Base interaction matrix: %s  nnz=%d  clip_val=%.4f",
        mat.shape, mat.nnz, clip_val,
    )
    return mat

In [54]:
def build_fpg_interaction_matrix(
    fpg_df:         pd.DataFrame,
    user_enc:       IDEncoder,
    item_enc:       IDEncoder,
    support_clip_pct: float = DEFAULT_SUPPORT_CLIP_PCT,
) -> coo_matrix:
    """
    Build the FPG-derived interaction matrix.

    FPG maps (group_id → items) with support counts.
    This function converts those into (user_idx → item_idx) interactions
    weighted by per-user-normalised log-support, then min-max normalised
    globally to the same [0, 1] range as the base matrix after log1p.

    Only (group_id, item) pairs where both IDs exist in the respective
    encoders contribute entries.  This includes:
      - Users who appear in both FPG and trainfile
      - FPG items whether or not they appear in trainfile
        (FPG-only items have index ≥ n_train_items but ≤ n_items_all)
    """
    logger.debug("build_fpg_interaction_matrix: %d FPG rows", len(fpg_df))

    with tqdm(total=5, desc="FPG interaction matrix",
              colour=TQDM_COLOUR, ncols=TQDM_NCOLS) as pbar:

        pbar.set_postfix_str("normalising support per group")
        # Per-user min-max normalise support
        fpg = fpg_df.copy()
        fpg["support_norm"] = fpg.groupby("group_id")["support"].transform(
            lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
        )
        pbar.update(1)

        pbar.set_postfix_str("log1p scaling support")
        clip_val = float(fpg["support"].quantile(support_clip_pct))
        fpg["support_clipped"] = fpg["support"].clip(upper=clip_val)
        #fpg["weight"] = np.log1p(fpg["support_clipped"]).astype(np.float32)
        fpg["weight"] = np.log1p(fpg["support_clipped"]) * fpg["support_norm"]
        pbar.update(1)

        pbar.set_postfix_str("encoding FPG user IDs (group_id)")
        user_idx = user_enc.transform(fpg["group_id"].values.astype(np.float64))
        pbar.update(1)

        pbar.set_postfix_str("encoding FPG item IDs")
        item_idx = item_enc.transform(fpg["items"].values.astype(str))
        pbar.update(1)

        pbar.set_postfix_str("building COO matrix")
        valid = (user_idx >= 0) & (item_idx >= 0)
        n_valid   = valid.sum()
        n_dropped = (~valid).sum()
        logger.debug(
            "FPG: %d valid entries, %d dropped (unknown user or item ID)",
            n_valid, n_dropped,
        )

        weights = fpg["weight"].values.astype(np.float32)
        mat = coo_matrix(
            (weights[valid], (user_idx[valid], item_idx[valid])),
            shape=(len(user_enc), len(item_enc)),
            dtype=np.float32,
        )
        mat = mat.tocsr()
        mat.sum_duplicates()
        mat = mat.tocoo()
        pbar.update(1)

    logger.info(
        "FPG interaction matrix: %s  nnz=%d  valid_pct=%.1f%%",
        mat.shape, mat.nnz, 100 * n_valid / max(len(fpg_df), 1),
    )
    return mat

In [55]:
def augment_interaction_matrix(
    base_mat:   coo_matrix,
    fpg_mat:    coo_matrix,
    cf_weight:  float = DEFAULT_CF_WEIGHT,
    fpg_weight: float = DEFAULT_FPG_WEIGHT,
) -> coo_matrix:
    """
    Produce the augmented interaction matrix:

        M_aug = cf_weight  × M_base
              + fpg_weight × M_fpg

    Both matrices are converted to CSR for element-wise ops,
    then back to COO for the Cython training kernels.

    The result is also globally min-max scaled to keep the
    training weight distribution stable regardless of blending ratio.
    """
    logger.debug(
        "augment_interaction_matrix: cf_w=%.2f  fpg_w=%.2f",
        cf_weight, fpg_weight,
    )

    with tqdm(total=3, desc="Augmenting matrix",
              colour=TQDM_COLOUR, ncols=TQDM_NCOLS) as pbar:

        pbar.set_postfix_str("blending CSR matrices")
        base_csr = base_mat.tocsr().astype(np.float32)
        fpg_csr  = fpg_mat.tocsr().astype(np.float32)
        augmented_csr = cf_weight * base_csr + fpg_weight * fpg_csr
        pbar.update(1)

        pbar.set_postfix_str("rescaling blended data")
        augmented_coo = augmented_csr.tocoo()
        if augmented_coo.data.max() > 0:
            #augmented_coo.data = (augmented_coo.data / augmented_coo.data.max()).astype(np.float32)
            p99 = np.percentile(augmented_coo.data, 99)
            augmented_coo.data = (np.clip(augmented_coo.data / p99, 0, 1)).astype(np.float32)
        pbar.update(1)

        pbar.set_postfix_str("deduplicating COO entries")
        # tocoo() after CSR arithmetic already sums duplicates, but
        # explicit sum_duplicates() guarantees correct COO form.
        augmented_coo = augmented_coo.tocsr().tocoo()
        pbar.update(1)

    logger.info(
        "Augmented matrix: %s  nnz=%d  data_range=[%.4f, %.4f]",
        augmented_coo.shape,
        augmented_coo.nnz,
        float(augmented_coo.data.min()) if augmented_coo.nnz else 0,
        float(augmented_coo.data.max()) if augmented_coo.nnz else 0,
    )
    return augmented_coo

## Encoder builder

In [56]:
def build_encoders(
    train_df: pd.DataFrame,
    fpg_df:   pd.DataFrame,
) -> Tuple[IDEncoder, IDEncoder]:
    """
    Build user and item encoders over the UNION of IDs from both sources.

    User universe  = train user_ids  ∪  FPG group_ids
    Item universe  = train product_ids ∪  FPG items

    This ensures CF can represent every entity seen in either dataset.
    """
    logger.debug("Building ID encoders")

    with tqdm(total=4, desc="Building encoders",
              colour=TQDM_COLOUR, ncols=TQDM_NCOLS) as pbar:

        pbar.set_postfix_str("collecting user IDs")
        train_users = set(train_df["user_id"].astype(np.float64).unique())
        fpg_users   = set(fpg_df["group_id"].astype(np.float64).unique())
        all_users   = train_users | fpg_users
        pbar.update(1)

        pbar.set_postfix_str("collecting item IDs")
        train_items = set(train_df["product_id"].astype(str).unique())
        fpg_items   = set(fpg_df["items"].astype(str).unique())
        all_items   = train_items | fpg_items
        pbar.update(1)

        pbar.set_postfix_str("fitting user encoder")
        user_enc = IDEncoder("user").fit(all_users)
        pbar.update(1)

        pbar.set_postfix_str("fitting item encoder")
        item_enc = IDEncoder("item").fit(all_items)
        pbar.update(1)

    logger.info(
        "Encoders: %d users (%d train + %d fpg-only)  |  "
        "%d items (%d train + %d fpg-only)",
        len(user_enc),
        len(train_users),
        len(fpg_users - train_users),
        len(item_enc),
        len(train_items),
        len(fpg_items - train_items),
    )
    return user_enc, item_enc

## FPG index for inference

In [57]:
def build_fpg_index(
    fpg_df:   pd.DataFrame,
    item_enc: IDEncoder,
) -> Dict[float, Dict[int, float]]:
    """
    Build a nested lookup dict for fast FPG scoring at inference time:
        { user_id_float → { item_idx_int → normalised_support_float } }

    Uses DuckDB to compute per-user normalised support scores.
    """
    logger.debug("Building FPG inference index …")

    with duckdb_connection() as con:
        con.register_dataframe("fpg", fpg_df)
        fpg_norm = con.query("""
            SELECT
                CAST(group_id AS DOUBLE)  AS user_id,
                items,
                support,
                CAST(support AS DOUBLE)
                    / NULLIF(MAX(support) OVER (PARTITION BY group_id), 0)
                    AS support_norm
            FROM fpg
        """)

    fpg_index: Dict[float, Dict[int, float]] = dict()

    with tqdm(total=len(fpg_norm), desc="Indexing FPG",
              colour=TQDM_COLOUR, ncols=TQDM_NCOLS, miniters=500) as pbar:

        for row in fpg_norm.itertuples(index=False):
            uid      = float(row.user_id)
            item_str = str(row.items)
            iidx     = item_enc._id2idx.get(item_str, -1)
            if iidx < 0:
                pbar.update(1)
                continue
            if uid not in fpg_index:
                fpg_index[uid] = dict()
            fpg_index[uid][iidx] = float(row.support_norm or 0.0)
            pbar.update(1)

    logger.info(
        "FPG index: %d users indexed, avg %.1f items/user",
        len(fpg_index),
        np.mean([len(v) for v in fpg_index.values()]) if fpg_index else 0,
    )
    return fpg_index

## model training

In [58]:
def train_model(
    augmented_mat:  coo_matrix,
    user_feat_mat:  csr_matrix,
    item_feat_mat:  csr_matrix,
    loss:           str   = "warp",
    no_components:  int   = 32,
    epochs:         int   = 20,
    num_threads:    int   = 4,
    learning_rate:  float = 0.05,
    item_alpha:     float = 1e-6,
    user_alpha:     float = 1e-6,
    max_sampled:    int   = 10,
    random_state:   int   = 42,
):
    """
    Train AryColBring on the augmented interaction matrix.

    Side-information feature matrices are passed so the model can learn
    content-aware embeddings on top of the collaborative signal.
    """
    logger.info(
        "Training AryColBring: loss=%s  no_components=%d  epochs=%d  threads=%d",
        loss, no_components, epochs, num_threads,
    )

    #try:
    #    from arycolbring import AryColBring
    #except ImportError as exc:
    #    raise RuntimeError(
    #        "arycolbring package not found.  "
    #        "Run: python setup.py build_ext --inplace"
    #    ) from exc

    model = AryColBring(
        no_components    = no_components,
        loss             = loss,
        learning_schedule = "adagrad",
        learning_rate    = learning_rate,
        item_alpha       = item_alpha,
        user_alpha       = user_alpha,
        max_sampled      = max_sampled,
        random_state     = random_state,
    )

    t0 = time.perf_counter()

    model.fit(
        augmented_mat,
        user_features = user_feat_mat,
        item_features = item_feat_mat,
        epochs        = epochs,
        num_threads   = num_threads,
        verbose       = True,
    )

    elapsed = time.perf_counter() - t0
    logger.info("Training complete in %.2f s", elapsed)
    return model

## Artefact Persistence

In [59]:
import cloudpickle
import pickle
import gzip
from pathlib import Path
from datetime import datetime
from typing import Dict
from scipy import sparse as sp

def save_artifacts(
    artifacts_dir : str,
    model         : object,
    user_enc      : IDEncoder,
    item_enc      : IDEncoder,
    fpg_index     : Dict,
    user_feat_mat : csr_matrix,
    item_feat_mat : csr_matrix,
    config        : Dict,
) -> str:
    """
    Save ALL artifacts into a SINGLE highly-compressed file.
    Output example:
        artifacts_20260424.pkl.gz
    """
    out = Path(artifacts_dir)
    out.mkdir(parents=True, exist_ok=True)
    logger.info("Saving artefacts to %s/", artifacts_dir)
    metadata = {"saved_at": datetime.utcnow().isoformat(),
                "artifact_version": "1.0",}
    artifacts = {
        "model"         : model,
        "user_encoder"  : user_enc,
        "item_encoder"  : item_enc,
        "fpg_index"     : fpg_index,
        "user_feat_mat" : user_feat_mat,
        "item_feat_mat" : item_feat_mat,
        "config"        : config,
        "metadata"      : metadata,
    }

    date_str = datetime.utcnow().strftime("%Y%m%d")
    final_path = out / f"artifacts_{date_str}.pkl.gz"
    tmp_path = final_path.with_suffix(".tmp")

    # Atomic write + high compression
    with gzip.open(tmp_path, "wb", compresslevel=9) as fh:
        cloudpickle.dump(artifacts,
            fh, protocol=pickle.HIGHEST_PROTOCOL)
    tmp_path.replace(final_path)
    logger.info("All artefacts saved to %s/", final_path)
    return final_path

## Training Execution

In [60]:
def main(
    train: str = DEFAULT_TRAIN_PATH,
    fpg: str = DEFAULT_FPG_PATH,
    artifacts: str = DEFAULT_ARTIFACTS,
    loss: str = "warp",
    no_components: int = 32,
    epochs: int = 20,
    threads: int = 4,
    learning_rate: float = 0.05,
    item_alpha: float = 1e-6,
    user_alpha: float = 1e-6,
    max_sampled: int = 10,
    cf_weight: float = DEFAULT_CF_WEIGHT,
    fpg_weight: float = DEFAULT_FPG_WEIGHT,
    random_state: int = 56,
) -> None:
    
    # ── Log Configuration ──────────────────────────────────────────
    # We create a local dict to mimic the old 'args' object behavior
    config_params = locals()
    
    logger.info("=" * 60)
    logger.info("Hybrid FPGrowth × AryColBring Pipeline")
    logger.info("=" * 60)
    logger.info("Config: %s", config_params)

    # ── 1. Load data ──────────────────────────────────────────────
    train_df, fpg_df = load_data(train, fpg)

    # ── 2. Build encoders over union of IDs ──────────────────────
    user_enc, item_enc = build_encoders(train_df, fpg_df)

    # ── 3. Base interaction matrix from trainfile ─────────────────
    base_mat = build_base_interaction_matrix(
        train_df  = train_df,
        user_enc  = user_enc,
        item_enc  = item_enc,
    )

    # ── 4. FPG augmentation matrix ────────────────────────────────
    fpg_mat = build_fpg_interaction_matrix(
        fpg_df   = fpg_df,
        user_enc = user_enc,
        item_enc = item_enc,
    )

    # ── 5. Blend into augmented matrix ────────────────────────────
    augmented_mat = augment_interaction_matrix(
        base_mat   = base_mat,
        fpg_mat    = fpg_mat,
        cf_weight  = cf_weight,
        fpg_weight = fpg_weight,
    )
    del base_mat, fpg_mat
    gc.collect()

    # ── 6. User & item feature matrices (GA4 side information) ────
    user_feat_mat = build_user_feature_matrix(train_df, user_enc)
    item_feat_mat = build_item_feature_matrix(train_df, item_enc)

    # ── 7. Build FPG index for fast inference blending ────────────
    fpg_index = build_fpg_index(fpg_df, item_enc)

    # ── 8. Train AryColBring ──────────────────────────────────────
    model = train_model(
        augmented_mat  = augmented_mat,
        user_feat_mat  = user_feat_mat,
        item_feat_mat  = item_feat_mat,
        loss           = loss,
        no_components  = no_components,
        epochs         = epochs,
        num_threads    = threads,
        learning_rate  = learning_rate,
        item_alpha     = item_alpha,
        user_alpha     = user_alpha,
        max_sampled    = max_sampled,
        random_state   = random_state,
    )

    # ── 9. Save artefacts ─────────────────────────────────────────
    save_config = {
        "cf_weight":     cf_weight,
        "fpg_weight":    fpg_weight,
        "loss":          loss,
        "no_components": no_components,
        "epochs":        epochs,
        "n_users":       len(user_enc),
        "n_items":       len(item_enc),
        "user_feat_cols": USER_FEAT_COLS,
        "item_feat_cols": ITEM_FEAT_COLS,
    }
    
    save_artifacts(
        artifacts_dir = artifacts,
        model         = model,
        user_enc      = user_enc,
        item_enc      = item_enc,
        fpg_index     = fpg_index,
        user_feat_mat = user_feat_mat,
        item_feat_mat = item_feat_mat,
        config        = save_config,
    )

    logger.info("Pipeline complete")
    return model

In [61]:
modelCF_FPG = main(
    train = mas_train,
    fpg = DataFPGpredict,
    artifacts = DEFAULT_ARTIFACTS)

2026-04-24 05:00:07,246 [INFO] hybrid_pipeline: ============================================================
2026-04-24 05:00:07,247 [INFO] hybrid_pipeline: Hybrid FPGrowth × AryColBring Pipeline
2026-04-24 05:00:07,248 [INFO] hybrid_pipeline: ============================================================
2026-04-24 05:00:07,248 [INFO] hybrid_pipeline: Config: {'train':             user_id      product_id  department_id  aisle_id  brand  \
0        66627256.0    GGOEGXXX1613             30         0      3   
1        66627256.0  GGCOGBJD157199             30         0      1   
2        66627256.0    GGCOGXXX1569             30         0      1   
3        66627256.0    GGOEGXXX1530             30         0      1   
4        66627256.0  GGCOGCBA161199             30         0      1   
...             ...             ...            ...       ...    ...   
1014421   3997498.0    GGCOGXXX1609             30         0      1   
1014422   3997498.0  GGCOGDHC161299             30         0 

In [63]:
dir(modelCF_FPG)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slotnames__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_check_finite',
 '_check_initialized',
 '_check_input_finite',
 '_check_test_train_intersections',
 '_construct_feature_matrices',
 '_epoch_iterator',
 '_epsilon',
 '_get_model_data',
 '_get_positives_lookup_matrix',
 '_initialize',
 '_item_alpha',
 '_k',
 '_learning_rate',
 '_learning_schedule',
 '_loss',
 '_max_sampled',
 '_n',
 '_no_components',
 '_process_sample_weight',
 '_random_state',
 '_reset_state',
 '_rho',
 '_run_epoch',
 '_to_cython_dtype',
 '_user_alpha',
 'fit',
 'fit_partial',
 'get_item_representations',
 'get_params',
 'get_user_representations',
 'is_fitted',
 'item_a